# Tool 9 — Spectral features: band power and aperiodic (1/f) components

Extracts, from the **clean epochs** written by `7_reject_manually` or `7bis_reject_automatically`,
the classic spectral features of a PSG recording:

- the **power spectral density** per epoch and channel (Welch or multitaper),
- the **aperiodic component** (offset, exponent) fitted with *specparam*, and the **periodic peaks**,
- the **band power** (delta, theta, alpha, sigma, beta, gamma) in several forms — absolute, relative,
  in dB, and corrected for the 1/f floor.

Everything is written as `.tsv` tables (epoch level and aggregated per sleep stage), one HTML report per
participant, and a database-level HTML report plus an Excel workbook.

**Nothing upstream is modified**: the epochs are read; the tool-6 preprocessing sidecar is used for
provenance only, and its absence is never fatal.

Work through the four sections in order: **1** pick the folders and scan, **2** set the parameters,
**3** choose the participants, **4** run.

In [ ]:
# =============================================================================
# Tool 9 - Spectral features: setup, constants and shared configuration
# =============================================================================
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import mne
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
matplotlib.rcParams['figure.max_open_warning'] = 0

# numpy 2 renamed trapz -> trapezoid; keep both working
_trapz = getattr(np, 'trapezoid', np.trapz)

# --- optional dependencies (all probed, never fatal at import time) -----------
try:
    from specparam import SpectralModel
    HAS_SPECPARAM = True
except Exception:
    HAS_SPECPARAM = False

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    HAS_LOWESS = True
except Exception:
    HAS_LOWESS = False

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

try:
    import openpyxl                     # noqa: F401  (pandas needs it to write .xlsx)
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False

# --- sleep stages ------------------------------------------------------------
AASM_STAGES = ['W', 'N1', 'N2', 'N3', 'R']

# Stage colours / geometry - duplicated from qc_rejected_epochs_lib (keep in sync)
BASE_STAGE_COLORS = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba',
                     'N3': '#6a51a3', 'R': '#c994c7'}
CUSTOM_STAGE_PALETTE = ['#8dd3c7', '#ffffb3', '#bebada', '#80b1d3',
                        '#fdb462', '#b3de69', '#fccde5', '#d9d9d9']

# --- frequency bands ---------------------------------------------------------
DEFAULT_BANDS = [('delta', 0.5, 4.0), ('theta', 4.0, 8.0), ('alpha', 8.0, 12.0),
                 ('sigma', 12.0, 16.0), ('beta', 16.0, 30.0), ('gamma', 30.0, 45.0)]

# --- exportable band-power measures ------------------------------------------
# (key, label, one-line description shown in the UI, default state, needs the 1/f fit)
MEASURES = [
    ('power_mean_uV2_Hz', 'Mean power density',
     'Mean of the PSD over the band (uV^2/Hz) - independent of the band width.',
     True, False),
    ('power_abs_uV2', 'Absolute power',
     'Integral of the PSD over the band (uV^2) - scales with the band width.',
     True, False),
    ('power_db', 'Power in dB',
     '10*log10 of the mean power density (dB/Hz) - the usual scale for statistics.',
     True, False),
    ('power_rel', 'Relative power',
     'Band absolute power divided by the absolute power of the whole PSD range - '
     'dimensionless, all bands sum to ~1.',
     True, False),
    ('power_ap_removed_db', 'Power above the 1/f fit (dB)',
     'Mean over the band of (PSD in dB - aperiodic fit in dB): how far the spectrum sits '
     'above the 1/f floor. Removes offset differences between subjects and stages.',
     True, True),
    ('power_ratio_over_aperiodic', 'Fold-change over the 1/f fit',
     'Mean over the band of (PSD / aperiodic fit) in linear space - a fold-change '
     '(2 = twice the 1/f floor). Always positive; not a re-expression of the dB measure '
     '(the mean of a ratio is not the ratio of the means).',
     False, True),
    ('power_rel_periodic', 'Relative periodic power',
     'Residual power max(PSD - aperiodic fit, 0) integrated over the band, divided by the same '
     'over the whole range. Like the two measures above it rests on the specparam fit, so it is '
     'only as good as that fit; the clipping at 0 makes it the most fragile of the three when the '
     'fit is poor - check the fit_ok rate in the report before using it.',
     False, True),
]
MEASURE_NEEDS_FIT = {k: needs for k, _, _, _, needs in MEASURES}

# Measures whose epoch -> stage average is reported in BOTH averaging spaces
# (see the note in Section 2): the dB ones. Everything else gets a plain mean.
DB_MEASURES = ['power_db', 'power_ap_removed_db']

# --- defaults ----------------------------------------------------------------
DEF_PSD = dict(method='welch', fmin=0.5, fmax=45.0, win_s=4.0, overlap_pct=50.0,
               window='hann', bandwidth=2.0)
# Peak model kept close to the specparam defaults (peak_width_limits=(0.5, 12), min_peak_height=0)
# rather than to tool 6's wider settings: a 20 Hz-wide "peak" can absorb part of the aperiodic
# slope, and a 0.3 threshold discards small oscillations whose power then leaks into the fit.
DEF_FIT = dict(fit_fmin=2.0, fit_fmax=45.0, aperiodic_mode='fixed',
               peak_width_min=0.5, peak_width_max=12.0, min_peak_height=0.1,
               max_n_peaks=8, peak_threshold=2.0, r2_min=0.90, mae_max=0.15)
DEF_SMOOTH = dict(enabled=True, lowess=True, median=True,
                  median_span_hz=3.0, lowess_span_hz=2.0)
DEF_MIN_EPOCHS = 20

# Below this many (epoch x channel) fits the serial loop beats the parallel one (worker
# start-up costs ~3 s on Windows, one fit ~3 ms) - measured, see fit_spectra.
PARALLEL_MIN_FITS = 5000
# A knee this large means specparam found no real bend and the parameter ran away.
KNEE_DEGENERATE = 1e4

# Output folder names (data vs reports split, toolkit convention)
DATA_DIRNAME = 'features_spectral'
REPORTS_DIRNAME = 'reports_features_spectral'

# Pipeline cost units for the per-participant progress bar (Load/PSD/Fit/Bands/Report)
COST_LOAD, COST_PSD, COST_FIT, COST_BANDS, COST_REPORT = 10, 8, 45, 5, 15


def stage_style(custom_stages):
    """Stage -> y position and colour for the hypnogram strips (AASM + custom below N3).
    Duplicated from qc_rejected_epochs_lib.custom_stage_style (keep in sync)."""
    stage_y = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    colors = dict(BASE_STAGE_COLORS)
    for i, cs in enumerate(custom_stages):
        stage_y[cs] = -1 - i
        colors[cs] = CUSTOM_STAGE_PALETTE[i % len(CUSTOM_STAGE_PALETTE)]
    ordered = sorted(stage_y.items(), key=lambda kv: kv[1])
    return stage_y, colors, [y for _, y in ordered], ['REM' if s == 'R' else s for s, _ in ordered]


def parse_custom_field(text):
    """Parse the comma-separated 'Custom stages' field into a clean, de-duplicated list."""
    seen = []
    for tok in str(text).split(','):
        tok = tok.strip()
        if tok and tok not in seen:
            seen.append(tok)
    return seen


def norm(p):
    """Normalised path/id string, used on BOTH sides of every comparison (Windows safety)."""
    return os.path.normcase(str(p))

In [ ]:
# =============================================================================
# Core computation - discovery, PSD, specparam fit, band measures, aggregation
# (pure functions: no widget is touched here)
# =============================================================================

# ---------------------------------------------------------------------------
# Discovery and provenance
# ---------------------------------------------------------------------------
def find_participants(root):
    """List participants under the chosen clean-epochs folder.

    Prefers the cleaned epochs written by tool 7 / 7bis (`*_clean-epo.fif`); when none is
    found, falls back to tool 6's un-rejected `*_all-epo.fif` so the tool still runs on a
    database where no rejection step was applied (the caller warns about it).
    Returns (participants, suffix) with suffix None when nothing was found."""
    root = Path(root)
    if not root.exists():
        return [], None
    for suffix in ('_clean-epo.fif', '_all-epo.fif'):
        found = sorted(root.rglob('*' + suffix), key=lambda p: p.name)
        if found:
            parts = []
            for f in found:
                parts.append({'file_id': f.name[:-len(suffix)], 'fif': f, 'folder': f.parent,
                              'subtree': f.parent.relative_to(root)})
            parts.sort(key=lambda d: d['file_id'])
            return parts, suffix
    return [], None


def detect_source(root, suffix):
    """Label the origin of the epochs, stored in every output as the `source` column."""
    if suffix == '_all-epo.fif':
        return 'raw'
    name = norm(Path(root).name)
    if 'manual' in name:
        return 'clean_manual'
    if 'auto' in name:
        return 'clean_auto'
    return 'clean'


def find_sidecar(file_id, fif_folder, clean_root, data_root):
    """Locate tool 6's {file_id}_preprocessing_params.json - it lives in `raw_epo/`, not next
    to the clean `.fif`. Searched beside the fif, then under the clean folder's parent
    (i.e. `derivatives/`, which holds `raw_epo*/`), then under the data folder.
    Returns a Path or None (absence is never fatal - the sidecar is provenance only)."""
    name = f'{file_id}_preprocessing_params.json'
    cand = Path(fif_folder) / name
    if cand.exists():
        return cand
    for root in [Path(clean_root).parent if clean_root else None,
                 Path(data_root) if data_root else None]:
        if root is None or not root.exists():
            continue
        try:
            hits = sorted(root.rglob(name))
        except Exception:
            hits = []
        if hits:
            return hits[0]
    return None


def load_sidecar(path):
    """Read tool 6's params sidecar - provenance only (filters, epoch length, PSD smoothing).
    Missing / unreadable -> found=False and every field None (non-fatal)."""
    info = {'found': False, 'epoch_length_s': None, 'filter': None, 'resample': None,
            'notch': None, 'psd_smoothing': None, 'methods_run': None,
            'custom_stages': [], 'raw': {}}
    if path is None or not Path(path).exists():
        return info
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        info['found'] = True
        info['raw'] = data
        info['epoch_length_s'] = data.get('epoch_length_s')
        info['filter'] = data.get('filter')
        info['resample'] = data.get('resample')
        info['notch'] = data.get('notch')
        info['psd_smoothing'] = data.get('psd_smoothing')
        info['methods_run'] = data.get('methods_run')
        info['custom_stages'] = list(data.get('custom_stages', []))
    except Exception:
        pass
    return info


def fit_settings_diff(cfg, side):
    """1/f settings that differ between THIS run and the tool-6 run that flagged the epochs.

    Tool 6 already rejects epochs on a 1/f criterion, so a low `fit_ok` rate here is worth
    explaining: most of the time it comes from the fit being recomputed with different settings
    (a different fit range, different thresholds, or smoothing that was added to tool 6 later)."""
    if not side.get('found'):
        return []
    rt = (side.get('raw') or {}).get('rejection_thresholds', {}) or {}
    fit = cfg['fit']
    diffs = []
    fr = rt.get('1f_fit_range_hz')
    if isinstance(fr, (list, tuple)) and len(fr) == 2:
        if (float(fr[0]), float(fr[1])) != (fit['fit_fmin'], fit['fit_fmax']):
            diffs.append(f"fit range {float(fr[0]):g}-{float(fr[1]):g} Hz in tool 6, "
                         f"{fit['fit_fmin']:g}-{fit['fit_fmax']:g} Hz here")
    if '1f_r2_min' in rt and float(rt['1f_r2_min']) != fit['r2_min']:
        diffs.append(f"R2 bound {float(rt['1f_r2_min']):g} in tool 6, {fit['r2_min']:g} here")
    if '1f_mae_max' in rt and float(rt['1f_mae_max']) != fit['mae_max']:
        diffs.append(f"MAE bound {float(rt['1f_mae_max']):g} in tool 6, {fit['mae_max']:g} here")
    t6_smooth = bool((side.get('psd_smoothing') or {}).get('enabled'))
    if t6_smooth != bool(cfg['smooth']['enabled']):
        diffs.append(f"PSD smoothing {'on' if t6_smooth else 'off'} in tool 6, "
                     f"{'on' if cfg['smooth']['enabled'] else 'off'} here")
    return diffs


def load_custom_stages(folder):
    """Read config_param/custom_stages.json under the data folder ([] when absent)."""
    if not folder:
        return []
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    if not path.exists():
        return []
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        stages = data.get('custom_stages', []) if isinstance(data, dict) else []
        return [str(s) for s in stages]
    except Exception:
        return []


def load_subject_info(path):
    """Read the optional participant-info table (.csv/.tsv). Returns (DataFrame, error)."""
    if not path:
        return None, ''
    try:
        sep = '\t' if norm(path).endswith('.tsv') else None
        df = pd.read_csv(path, sep=sep, engine='python')
        return df, ''
    except Exception as exc:
        return None, str(exc)


def assign_thirds(epoch_idx, stages):
    """Split the sleep period into three equal spans of epoch index (draft feature).

    The sleep period runs from the first to the last epoch scored as anything but W; it is
    divided into three equal spans of the ORIGINAL epoch index, so the thirds follow real
    time even though the rejected epochs are missing from the file. Epochs outside are
    labelled `pre` / `post`. NREM-REM cycle detection is deliberately not attempted."""
    epoch_idx = np.asarray(epoch_idx)
    stages = np.asarray(stages)
    out = np.array(['none'] * len(stages), dtype=object)
    sleep = np.where(stages != 'W')[0]
    if len(sleep) == 0:
        return out
    lo, hi = float(epoch_idx[sleep[0]]), float(epoch_idx[sleep[-1]])
    span = max((hi - lo + 1) / 3.0, 1e-9)
    for i, e in enumerate(epoch_idx):
        if e < lo:
            out[i] = 'pre'
        elif e > hi:
            out[i] = 'post'
        else:
            out[i] = 'T%d' % (min(2, int((e - lo) // span)) + 1)
    return out


# ---------------------------------------------------------------------------
# PSD
# ---------------------------------------------------------------------------
def compute_psd_array(epochs, method='welch', fmin=0.5, fmax=45.0, win_s=4.0,
                      overlap_pct=50.0, window='hann', bandwidth=2.0):
    """Power spectral density per (epoch, channel), in uV^2/Hz.

    Welch: sliding window of `win_s` seconds with `overlap_pct` overlap (4 s -> 0.25 Hz
    resolution, the sleep-research convention; the rule of thumb is a resolution fine enough
    for ~2 cycles of the lowest frequency of interest). Multitaper: `bandwidth` Hz smoothing.
    fmax is capped at the Nyquist frequency and the window at the epoch length; both are
    reported back in `notes` so the run log can warn. Returns (freqs, psds_uV2, notes)."""
    notes = []
    sf = float(epochs.info['sfreq'])
    epoch_len_s = len(epochs.times) / sf
    fmax_eff = min(float(fmax), sf / 2 - 0.5)
    if fmax_eff < fmax:
        notes.append(f'fmax capped to {fmax_eff:.1f} Hz (Nyquist at sfreq={sf:g} Hz)')
    if method == 'multitaper':
        # normalization='full' is REQUIRED: MNE defaults to 'length', which is not a power
        # density - it is larger by a factor of sfreq (x256 = +24 dB at 256 Hz) and, worse,
        # the factor follows the sampling rate, so recordings at 256 and 512 Hz would not be
        # comparable. With 'full' the multitaper PSD matches Welch to within ~0.6 dB.
        spec = epochs.compute_psd(method='multitaper', fmin=fmin, fmax=fmax_eff,
                                  bandwidth=bandwidth, normalization='full', verbose=False)
    else:
        win_eff = min(float(win_s), epoch_len_s)
        if win_eff < win_s:
            notes.append(f'Welch window clipped to the epoch length ({win_eff:g} s)')
        n_per_seg = max(int(round(win_eff * sf)), 8)
        n_overlap = int(n_per_seg * float(overlap_pct) / 100.0)
        spec = epochs.compute_psd(method='welch', fmin=fmin, fmax=fmax_eff,
                                  n_fft=n_per_seg, n_overlap=n_overlap, n_per_seg=n_per_seg,
                                  window=window, verbose=False)
    return spec.freqs, spec.get_data() * 1e12, notes      # V^2/Hz -> uV^2/Hz


# --- PSD smoothing (duplicated VERBATIM from 6_preprocessing_voila / -------------
# --- qc_rejected_epochs_lib - keep in sync). Applied to the FIT COPY only. -------
def smooth_psd_median(psds_uV2, freqs, span_hz=3.0):
    """Running-median smooth of each (epoch, channel) linear PSD along frequency.
    Reproduces oscip.smooth_spectrum_median (MATLAB movmedian over a span given in Hz):
    a centred median filter that removes narrow spikes (residual line-noise harmonics,
    single-bin artefacts) BEFORE the LOWESS mean smoothing."""
    freqs = np.asarray(freqs)
    if len(freqs) < 4:
        return psds_uV2
    freq_res = float(np.median(np.diff(freqs)))
    span_pts = max(3, int(round(span_hz / freq_res)))
    out = np.array(psds_uV2, dtype=float, copy=True)
    n_ep, n_ch, _ = out.shape
    for ei in range(n_ep):
        for ci in range(n_ch):
            try:
                out[ei, ci] = pd.Series(out[ei, ci]).rolling(
                    window=span_pts, center=True, min_periods=1).median().to_numpy()
            except Exception:
                pass
    return out


def smooth_psd_lowess(psds_uV2, freqs, span_hz=2.0):
    """LOWESS-smooth each (epoch, channel) linear PSD along frequency.
    Reproduces oscip.smooth_spectrum (MATLAB non-robust 'lowess', local linear regression)
    over a span given in Hz, applied to the LINEAR power. Non-fatal per spectrum."""
    freqs = np.asarray(freqs)
    if len(freqs) < 4 or not HAS_LOWESS:
        return psds_uV2
    freq_res = float(np.median(np.diff(freqs)))
    span_pts = max(3, int(round(span_hz / freq_res)))
    frac = min(1.0, span_pts / len(freqs))
    out = np.array(psds_uV2, dtype=float, copy=True)
    n_ep, n_ch, _ = out.shape
    for ei in range(n_ep):
        for ci in range(n_ch):
            try:
                out[ei, ci] = lowess(out[ei, ci], freqs, frac=frac, it=0, return_sorted=False)
            except Exception:
                pass
    return out


def smooth_for_fit(psds_uV2, freqs, smooth_cfg):
    """Apply the tool-6 smoothing (median first, then LOWESS - the oscip order) to the copy that
    feeds specparam. Each filter can be used alone. The band powers are ALWAYS computed on the
    unsmoothed PSD."""
    if not smooth_cfg.get('enabled'):
        return psds_uV2
    out = psds_uV2
    if smooth_cfg.get('median'):
        out = smooth_psd_median(out, freqs, span_hz=float(smooth_cfg.get('median_span_hz', 3.0)))
    if smooth_cfg.get('lowess', True):
        out = smooth_psd_lowess(out, freqs, span_hz=float(smooth_cfg.get('lowess_span_hz', 2.0)))
    return out


# ---------------------------------------------------------------------------
# Aperiodic / periodic decomposition (specparam)
# ---------------------------------------------------------------------------
def aperiodic_log10(freqs, mode, params):
    """Aperiodic component in log10(power), evaluated on ANY frequency axis.

    Evaluating the model analytically (rather than reading specparam's fitted curve) lets the
    aperiodic component be extrapolated below the fit's lower bound, so the delta band can be
    corrected too - at the cost of trusting the model outside the fitted range."""
    freqs = np.asarray(freqs, dtype=float)
    if mode == 'knee':
        offset, knee, exponent = params[0], params[1], params[2]
        return offset - np.log10(np.maximum(knee + freqs ** exponent, 1e-30))
    offset, exponent = params[0], params[1]
    return offset - exponent * np.log10(freqs)


def _fit_one_epoch(freqs_fit, psd_ep_fit, cfg, ei=0, peaks=None):
    """specparam fit of every channel of ONE epoch.
    psd_ep_fit: (n_ch, n_freqs_fit). Returns the (n_ch, 6) parameter array and appends
    (epoch, channel, peak_idx, n_peaks, cf, pw, bw) rows to `peaks`."""
    n_ch = psd_ep_fit.shape[0]
    out = np.full((n_ch, 6), np.nan)          # offset, knee, exponent, r2, mae, n_peaks
    if peaks is None:
        peaks = []
    if not HAS_SPECPARAM:
        return out, peaks
    for ci in range(n_ch):
        psd = psd_ep_fit[ci]
        if not np.all(np.isfinite(psd)) or np.any(psd <= 0):
            continue                          # dead / clipped spectrum: leave NaN
        try:
            sm = SpectralModel(peak_width_limits=[cfg['peak_width_min'], cfg['peak_width_max']],
                               aperiodic_mode=cfg['aperiodic_mode'],
                               min_peak_height=cfg['min_peak_height'],
                               max_n_peaks=int(cfg['max_n_peaks']),
                               peak_threshold=cfg['peak_threshold'], verbose=False)
            sm.fit(freqs_fit, psd)
            ap = np.asarray(sm.get_params('aperiodic'), dtype=float).ravel()
            if cfg['aperiodic_mode'] == 'knee':
                offset, knee, exponent = ap[0], ap[1], ap[2]
            else:
                offset, knee, exponent = ap[0], np.nan, ap[1]
            pk = np.asarray(sm.get_params('peak'), dtype=float)
            pk = pk.reshape(-1, 3) if pk.size else np.zeros((0, 3))
            out[ci] = [offset, knee, exponent,
                       float(sm.get_metrics('gof', 'squared')),
                       float(sm.get_metrics('error', 'mae')), len(pk)]
            for pi in range(len(pk)):
                peaks.append((ei, ci, pi + 1, len(pk), pk[pi, 0], pk[pi, 1], pk[pi, 2]))
        except Exception:
            continue                          # failed fit stays NaN (counted as fit_ok=False)
    return out, peaks


def _fit_chunk(freqs_fit, psd_chunk, cfg, ei0):
    """Fit a contiguous block of epochs - the unit of work handed to each parallel worker.
    Chunking (rather than one task per epoch) is what makes the parallel path actually
    faster: a single epoch is only a few fits, far too little to pay for the dispatch."""
    n_ep = psd_chunk.shape[0]
    arr = np.full((n_ep, psd_chunk.shape[1], 6), np.nan)
    peaks = []
    for k in range(n_ep):
        arr[k], _ = _fit_one_epoch(freqs_fit, psd_chunk[k], cfg, ei=ei0 + k, peaks=peaks)
    return ei0, arr, peaks


def fit_spectra(freqs, psds_uV2, cfg, n_jobs=1, progress=None):
    """specparam fit of every (epoch, channel) spectrum.

    Returns (res, peak_rows) where res holds (n_ep, n_ch) arrays for offset / knee /
    exponent / r2 / mae / n_peaks / fit_ok, and peak_rows is a list of
    (epoch_i, channel_i, peak_idx, n_peaks, center_freq, power, bandwidth).
    The parallel path is exact - it calls the same per-epoch function as the serial one -
    and any failure falls back to the serial loop."""
    n_ep, n_ch, _ = psds_uV2.shape
    fmask = (np.asarray(freqs) >= cfg['fit_fmin']) & (np.asarray(freqs) <= cfg['fit_fmax'])
    freqs_fit = np.asarray(freqs)[fmask]
    psds_fit = psds_uV2[:, :, fmask]
    params = np.full((n_ep, n_ch, 6), np.nan)
    peak_rows = []

    # Spawning workers costs ~3 s on Windows, so the parallel path only pays off on a big
    # job (measured 2.8 ms/fit: below ~5000 fits the serial loop is simply faster).
    done = False
    if n_jobs != 1 and HAS_JOBLIB and n_ep > 8 and n_ep * n_ch >= PARALLEL_MIN_FITS:
        try:
            n_workers = os.cpu_count() or 4 if n_jobs < 0 else n_jobs
            chunk = max(8, int(np.ceil(n_ep / max(1, n_workers))))
            starts = list(range(0, n_ep, chunk))
            results = Parallel(n_jobs=n_jobs)(
                delayed(_fit_chunk)(freqs_fit, psds_fit[s:s + chunk], cfg, s) for s in starts)
            for ei0, arr, peaks in results:
                params[ei0:ei0 + len(arr)] = arr
                peak_rows.extend(peaks)
            done = True
        except Exception:
            params[:] = np.nan
            peak_rows = []
            done = False                      # fall back to the serial loop
    if not done:
        for ei in range(n_ep):
            params[ei], _ = _fit_one_epoch(freqs_fit, psds_fit[ei], cfg, ei=ei, peaks=peak_rows)
            if progress is not None and (ei % 25 == 0 or ei == n_ep - 1):
                progress(ei + 1, n_ep)

    res = {'offset': params[:, :, 0], 'knee': params[:, :, 1], 'exponent': params[:, :, 2],
           'r2': params[:, :, 3], 'mae': params[:, :, 4], 'n_peaks': params[:, :, 5]}
    res['fit_ok'] = (np.isfinite(res['r2']) & np.isfinite(res['mae'])
                     & (res['r2'] >= cfg['r2_min']) & (res['mae'] <= cfg['mae_max']))
    return res, peak_rows


def aperiodic_psd_log10(freqs, res, mode):
    """Aperiodic model evaluated on the full PSD axis -> (n_ep, n_ch, n_freqs) in log10 power."""
    n_ep, n_ch = res['offset'].shape
    out = np.full((n_ep, n_ch, len(freqs)), np.nan)
    for ei in range(n_ep):
        for ci in range(n_ch):
            off, exp_, knee = res['offset'][ei, ci], res['exponent'][ei, ci], res['knee'][ei, ci]
            if not (np.isfinite(off) and np.isfinite(exp_)):
                continue
            if mode == 'knee' and not np.isfinite(knee):
                continue
            out[ei, ci] = aperiodic_log10(freqs, mode, (off, knee, exp_) if mode == 'knee'
                                          else (off, exp_))
    return out


# ---------------------------------------------------------------------------
# Band measures
# ---------------------------------------------------------------------------
def compute_band_measures(freqs, psds_uV2, ap_log10, bands, measures_on):
    """Band power for every selected measure.

    Returns {band_name: {measure: (n_ep, n_ch) array}}. Non-positive PSD bins (dead or
    clipped channels) become NaN before any log, so a bad channel yields NaN rather than -inf.
    Bin selection is half-open [fmin, fmax) so adjacent bands never share a bin, and the
    integral is a rectangular sum (each bin stands for a `df`-wide slab). With a 0.25 Hz
    resolution that gives each band its exact width (delta 0.5-4 Hz -> 14 bins x 0.25 = 3.5 Hz)
    and, when the bands tile the PSD range without gaps, relative powers that sum to 1 -
    which a trapezoidal rule over the band's bins alone would not do."""
    freqs = np.asarray(freqs, dtype=float)
    df = float(np.median(np.diff(freqs))) if len(freqs) > 1 else 1.0
    psd = np.where(psds_uV2 > 0, psds_uV2, np.nan)
    psd_db = 10.0 * np.log10(psd)
    ap_lin = 10.0 ** ap_log10 if ap_log10 is not None else None
    ap_db = 10.0 * ap_log10 if ap_log10 is not None else None
    ratio = psd / ap_lin if ap_lin is not None else None
    residual = np.maximum(psd - ap_lin, 0.0) if ap_lin is not None else None

    total_abs = np.nansum(psd, axis=2) * df
    total_res = np.nansum(residual, axis=2) * df if residual is not None else None

    out = {}
    for name, lo, hi in bands:
        m = (freqs >= lo) & (freqs < hi)
        if m.sum() < 1:
            continue                          # band outside the PSD range: skipped upstream
        vals = {}
        if 'power_mean_uV2_Hz' in measures_on or 'power_db' in measures_on:
            mean_lin = np.nanmean(psd[:, :, m], axis=2)
            if 'power_mean_uV2_Hz' in measures_on:
                vals['power_mean_uV2_Hz'] = mean_lin
            if 'power_db' in measures_on:
                vals['power_db'] = 10.0 * np.log10(np.where(mean_lin > 0, mean_lin, np.nan))
        if 'power_abs_uV2' in measures_on or 'power_rel' in measures_on:
            abs_pow = np.nansum(psd[:, :, m], axis=2) * df
            if 'power_abs_uV2' in measures_on:
                vals['power_abs_uV2'] = abs_pow
            if 'power_rel' in measures_on:
                vals['power_rel'] = abs_pow / np.where(total_abs > 0, total_abs, np.nan)
        if 'power_ap_removed_db' in measures_on and ap_db is not None:
            vals['power_ap_removed_db'] = np.nanmean(psd_db[:, :, m] - ap_db[:, :, m], axis=2)
        if 'power_ratio_over_aperiodic' in measures_on and ratio is not None:
            vals['power_ratio_over_aperiodic'] = np.nanmean(ratio[:, :, m], axis=2)
        if 'power_rel_periodic' in measures_on and residual is not None:
            res_band = np.nansum(residual[:, :, m], axis=2) * df
            vals['power_rel_periodic'] = res_band / np.where(total_res > 0, total_res, np.nan)
        out[name] = vals
    return out


# ---------------------------------------------------------------------------
# Long-format tables (epoch level)
# ---------------------------------------------------------------------------
def build_bandpower_epoch(file_id, source, meta, ch_names, band_vals, measures_on, thirds=None):
    """Long table: one row per (epoch, channel, band)."""
    rows = []
    epoch_idx = meta['epoch_idx'].values
    stages = meta['stage'].astype(str).values
    for band, vals in band_vals.items():
        for ci, ch in enumerate(ch_names):
            for ei in range(len(stages)):
                row = {'file_id': file_id, 'source': source, 'epoch_idx': int(epoch_idx[ei]),
                       'stage': stages[ei], 'channel': ch, 'band': band}
                if thirds is not None:
                    row['third'] = thirds[ei]
                for meas in measures_on:
                    if meas in vals:
                        row[meas] = float(vals[meas][ei, ci])
                rows.append(row)
    return pd.DataFrame(rows)


def build_aperiodic_epoch(file_id, source, meta, ch_names, res, thirds=None):
    """Long table: one row per (epoch, channel) with the aperiodic parameters and fit quality."""
    rows = []
    epoch_idx = meta['epoch_idx'].values
    stages = meta['stage'].astype(str).values
    for ci, ch in enumerate(ch_names):
        for ei in range(len(stages)):
            row = {'file_id': file_id, 'source': source, 'epoch_idx': int(epoch_idx[ei]),
                   'stage': stages[ei], 'channel': ch}
            if thirds is not None:
                row['third'] = thirds[ei]
            row.update({'offset': float(res['offset'][ei, ci]),
                        'knee': float(res['knee'][ei, ci]),
                        'exponent': float(res['exponent'][ei, ci]),
                        'r2': float(res['r2'][ei, ci]),
                        'mae': float(res['mae'][ei, ci]),
                        'n_peaks': float(res['n_peaks'][ei, ci]),
                        'fit_ok': bool(res['fit_ok'][ei, ci])})
            rows.append(row)
    return pd.DataFrame(rows)


def build_peaks_table(file_id, source, meta, ch_names, peak_rows):
    """Long table: one row per specparam peak (centre frequency, power, bandwidth)."""
    epoch_idx = meta['epoch_idx'].values
    stages = meta['stage'].astype(str).values
    rows = []
    for ei, ci, pi, npk, cf, pw, bw in peak_rows:
        rows.append({'file_id': file_id, 'source': source, 'epoch_idx': int(epoch_idx[ei]),
                     'stage': stages[ei], 'channel': ch_names[ci], 'peak_idx': int(pi),
                     'n_peaks': int(npk), 'center_freq_hz': float(cf),
                     'peak_power': float(pw), 'bandwidth_hz': float(bw)})
    return pd.DataFrame(rows, columns=['file_id', 'source', 'epoch_idx', 'stage', 'channel',
                                       'peak_idx', 'n_peaks', 'center_freq_hz',
                                       'peak_power', 'bandwidth_hz'])


# ---------------------------------------------------------------------------
# Aggregation (epoch -> stage)
# ---------------------------------------------------------------------------
def stage_masks(stages, stages_sel):
    """Ordered {stage: boolean epoch mask} for the selected stages.
    Groupings such as NREM or whole-night are deliberately left to the user: they are one
    groupby away in the exported tables, and computing them here doubled the table size."""
    stages = np.asarray(stages)
    return {st: (stages == st) for st in stages_sel}


def _agg_stats(values, min_epochs):
    """mean / median / sd over the epochs of one cell; NaN when too few epochs."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < max(1, min_epochs):
        return np.nan, np.nan, np.nan
    return float(np.mean(v)), float(np.median(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0


def aggregate_bandpower(file_id, source, band_vals, ch_names, masks, measures_on,
                        min_epochs, spaces=('log',), fit_ok=None, exclude_bad_fit=False):
    """Aggregate the per-epoch band measures to one row per (stage, channel, band).

    `spaces` picks how the dB measures are averaged over the epochs of a stage: 'log' = mean of
    the per-epoch dB (what a per-epoch statistical model sees, robust to one loud epoch), 'lin' =
    dB of the mean in linear space (preserves total power, but a few loud epochs dominate). The
    two are exported side by side when both are asked for, suffixed `_from_log` / `_from_lin`.
    The linear measures get a plain mean/median/sd. Cells with fewer than `min_epochs` epochs are
    written as NaN, with n_epochs kept, so a shortage is visible instead of silent."""
    rows = []
    for label, mask in masks.items():
        for ci, ch in enumerate(ch_names):
            use = mask.copy()
            if exclude_bad_fit and fit_ok is not None:
                use = use & fit_ok[:, ci]
            n_ep = int(use.sum())
            n_ok = int((mask & fit_ok[:, ci]).sum()) if fit_ok is not None else np.nan
            for band, vals in band_vals.items():
                row = {'file_id': file_id, 'source': source, 'stage': label, 'channel': ch,
                       'band': band, 'n_epochs': n_ep, 'n_fit_ok': n_ok}
                for meas in measures_on:
                    if meas not in vals:
                        continue
                    v = vals[meas][use, ci]
                    mean, med, sd = _agg_stats(v, min_epochs)
                    if meas not in DB_MEASURES:
                        row[f'{meas}_mean'] = mean
                        row[f'{meas}_median'] = med
                        row[f'{meas}_sd'] = sd
                        continue
                    if 'log' in spaces:
                        row[f'{meas}_from_log_mean'] = mean
                        row[f'{meas}_from_log_median'] = med
                        row[f'{meas}_from_log_sd'] = sd
                    if 'lin' in spaces:
                        lin = 10.0 ** (np.asarray(v, dtype=float) / 10.0)
                        lm, lmd, _ = _agg_stats(lin, min_epochs)
                        row[f'{meas}_from_lin_mean'] = (10.0 * np.log10(lm)
                                                        if np.isfinite(lm) and lm > 0 else np.nan)
                        row[f'{meas}_from_lin_median'] = (10.0 * np.log10(lmd)
                                                          if np.isfinite(lmd) and lmd > 0 else np.nan)
                rows.append(row)
    return pd.DataFrame(rows)


def aggregate_aperiodic(file_id, source, res, ch_names, masks, min_epochs,
                        exclude_bad_fit=False):
    """One row per (stage, channel) with the aperiodic parameters and the fit quality."""
    rows = []
    for label, mask in masks.items():
        for ci, ch in enumerate(ch_names):
            use = mask.copy()
            if exclude_bad_fit:
                use = use & res['fit_ok'][:, ci]
            row = {'file_id': file_id, 'source': source, 'stage': label, 'channel': ch,
                   'n_epochs': int(use.sum()),
                   'n_fit_ok': int((mask & res['fit_ok'][:, ci]).sum())}
            for key in ('offset', 'exponent', 'knee', 'r2', 'mae', 'n_peaks'):
                mean, med, sd = _agg_stats(res[key][use, ci], min_epochs)
                row[f'{key}_mean'], row[f'{key}_median'], row[f'{key}_sd'] = mean, med, sd
            rows.append(row)
    return pd.DataFrame(rows)


def aggregate_psd(file_id, source, freqs, psds_uV2, ap_log10, ch_names, masks, min_epochs,
                  spaces=('log',)):
    """Mean PSD per (stage, channel, frequency) - lets the figures be redrawn without recomputing
    anything. The dB columns follow the selected averaging space(s); the linear mean is always
    kept, since it is the raw material every other column derives from.

    The `_sd` columns are the standard deviation ACROSS EPOCHS, so the +/- SEM band of the report
    figures can be reproduced from this table alone: sem = sd / sqrt(n_epochs)."""
    psd = np.where(psds_uV2 > 0, psds_uV2, np.nan)
    psd_db = 10.0 * np.log10(psd)
    ap_removed = psd_db - 10.0 * ap_log10 if ap_log10 is not None else None
    rows = []
    for label, mask in masks.items():
        n_ep = int(mask.sum())
        if n_ep < max(1, min_epochs):
            continue
        with np.errstate(invalid='ignore'):
            lin = np.nanmean(psd[mask], axis=0)                 # (n_ch, n_freqs)
            lin_sd = np.nanstd(psd[mask], axis=0, ddof=1)
            log = np.nanmean(psd_db[mask], axis=0)
            log_sd = np.nanstd(psd_db[mask], axis=0, ddof=1)
            apr = np.nanmean(ap_removed[mask], axis=0) if ap_removed is not None else None
            apr_sd = np.nanstd(ap_removed[mask], axis=0, ddof=1) if ap_removed is not None else None
        for ci, ch in enumerate(ch_names):
            for fi, fr in enumerate(freqs):
                row = {'file_id': file_id, 'source': source, 'stage': label, 'channel': ch,
                       'freq_hz': float(fr), 'n_epochs': n_ep,
                       'psd_uV2_Hz_from_lin': float(lin[ci, fi]),
                       'psd_uV2_Hz_sd': float(lin_sd[ci, fi])}
                if 'log' in spaces:
                    row['psd_db_from_log'] = float(log[ci, fi])
                    row['psd_db_from_log_sd'] = float(log_sd[ci, fi])
                if 'lin' in spaces:
                    row['psd_db_from_lin'] = (float(10.0 * np.log10(lin[ci, fi]))
                                              if np.isfinite(lin[ci, fi]) and lin[ci, fi] > 0
                                              else np.nan)
                if apr is not None:
                    row['psd_ap_removed_db'] = float(apr[ci, fi])
                    row['psd_ap_removed_db_sd'] = float(apr_sd[ci, fi])
                rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# =============================================================================
# Figures and HTML tables (participant level and database level)
# =============================================================================
def _fig_width(n_ch):
    return min(5.0 * max(n_ch, 1), 20.0)


def _axes_row(n_ch, height=4.2, sharey=True):
    fig, axes = plt.subplots(1, max(n_ch, 1), figsize=(_fig_width(n_ch), height), sharey=sharey)
    return fig, np.atleast_1d(axes)


def stage_curves(values, masks, min_epochs):
    """Mean + SEM across epochs of a (n_ep, n_ch, n_freqs) array, per stage."""
    out = {}
    for label, m in masks.items():
        n = int(m.sum())
        if n < max(1, min_epochs):
            continue
        sub = values[m]
        with np.errstate(invalid='ignore'):
            mean = np.nanmean(sub, axis=0)
            sem = np.nanstd(sub, axis=0, ddof=1) / np.sqrt(max(n, 1))
        out[label] = (mean, sem, n)
    return out


# --- 1. hypnogram of the epochs that survived the rejection ------------------
def plot_clean_hypnogram(meta, custom_stages, file_id, n_total=None):
    """Stage of every surviving epoch against its ORIGINAL epoch index, so the epochs removed
    by tools 6/7/7bis show up as gaps, plus the clean-epoch count per stage."""
    stage_y, colors, ytick_pos, ytick_lab = stage_style(custom_stages)
    epoch_idx = meta['epoch_idx'].values
    stages = meta['stage'].astype(str).values
    fig, axes = plt.subplots(1, 2, figsize=(15, 3.2), gridspec_kw={'width_ratios': [3, 1]})

    ax = axes[0]
    y = np.array([stage_y.get(s, np.nan) for s in stages], dtype=float)
    ax.scatter(epoch_idx, y, s=6, c=[colors.get(s, '#888888') for s in stages])
    ax.set_yticks(ytick_pos)
    ax.set_yticklabels(ytick_lab)
    ax.set_xlabel('Epoch index in the recording')
    kept = len(stages)
    total = n_total if n_total else (int(epoch_idx.max()) + 1 if len(epoch_idx) else 0)
    ax.set_title(f'{file_id} - clean epochs: {kept} kept out of {total} '
                 f'({100.0 * kept / max(total, 1):.0f} %); gaps = removed epochs')
    ax.grid(alpha=0.2)

    ax = axes[1]
    order = [s for s in AASM_STAGES + list(custom_stages) if s in set(stages)]
    counts = [int((stages == s).sum()) for s in order]
    ax.bar(range(len(order)), counts, color=[colors.get(s, '#888888') for s in order])
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(['REM' if s == 'R' else s for s in order])
    ax.set_ylabel('clean epochs')
    ax.set_title('Clean epochs per stage')
    for i, c in enumerate(counts):
        ax.text(i, c, str(c), ha='center', va='bottom', fontsize=8)
    fig.tight_layout()
    return fig


# --- 2 & 3. mean PSD per stage (raw and aperiodic-removed) -------------------
def _curve_from_log(values_db, mask):
    """Average per-epoch dB values (log space); returns (mean, lo, hi, n) in dB."""
    sub = values_db[mask]
    n = int(mask.sum())
    with np.errstate(invalid='ignore'):
        mean = np.nanmean(sub, axis=0)
        sem = np.nanstd(sub, axis=0, ddof=1) / np.sqrt(max(n, 1))
    return mean, mean - sem, mean + sem, n


def _curve_from_lin(values_lin, mask):
    """Average linear power, then convert to dB; returns (mean, lo, hi, n) in dB."""
    sub = values_lin[mask]
    n = int(mask.sum())
    with np.errstate(invalid='ignore'):
        mean = np.nanmean(sub, axis=0)
        sem = np.nanstd(sub, axis=0, ddof=1) / np.sqrt(max(n, 1))

    def to_db(x):
        return 10.0 * np.log10(np.where(x > 0, x, np.nan))

    return to_db(mean), to_db(mean - sem), to_db(mean + sem), n


def plot_psd_per_stage(freqs, masks, ch_names, custom_stages, min_epochs, title, ylabel,
                       values_db=None, values_lin=None):
    """One subplot per channel; one curve per stage with a +/- SEM band.

    `values_db` holds per-epoch values already in dB (averaged in LOG space); `values_lin` holds
    per-epoch LINEAR power (averaged, then converted to dB). Whichever is available is drawn
    solid; when both are, the second is dashed behind so the gap between the two averaging
    spaces is visible on the figure itself."""
    _, colors, _, _ = stage_style(custom_stages)
    primary = ('log', values_db) if values_db is not None else ('lin', values_lin)
    secondary = ('lin', values_lin) if (values_db is not None and values_lin is not None) else None
    fig, axes = _axes_row(len(ch_names))
    for ci, ch in enumerate(ch_names):
        ax = axes[ci]
        for stage, mask in masks.items():
            if int(mask.sum()) < max(1, min_epochs):
                continue
            col = colors.get(stage, '#888888')
            fn = _curve_from_log if primary[0] == 'log' else _curve_from_lin
            mean, lo, hi, n = fn(primary[1], mask)
            ax.plot(freqs, mean[ci], color=col, label=f'{stage} (n={n})')
            ax.fill_between(freqs, lo[ci], hi[ci], color=col, alpha=0.25)
            if secondary is not None:
                alt, _, _, _ = _curve_from_lin(secondary[1], mask)
                ax.plot(freqs, alt[ci], color=col, ls='--', lw=0.8, alpha=0.7)
        ax.set_title(ch)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_xlim(freqs[0], freqs[-1])
        ax.grid(alpha=0.2)
        if ci == 0:
            ax.set_ylabel(ylabel)
            ax.legend(fontsize=8)
    fig.suptitle(title + (' (solid: log-space average, dashed: linear-space average)'
                          if secondary is not None else ''))
    fig.tight_layout()
    return fig


# --- 4. log-log PSD + aperiodic fit ------------------------------------------
def plot_loglog_with_fit(freqs, psds_uV2, ap_log10, masks, ch_names, custom_stages,
                         min_epochs, fit_range, res=None):
    """Mean PSD per stage in log-log with the mean aperiodic fit dashed on top - the visual
    check of the fit, and of whether a `knee` is needed (a real bend in the straight line)."""
    _, colors, _, _ = stage_style(custom_stages)
    psd_db = 10.0 * np.log10(np.where(psds_uV2 > 0, psds_uV2, np.nan))
    curves = stage_curves(psd_db, masks, min_epochs)
    ap_curves = stage_curves(10.0 * ap_log10, masks, min_epochs) if ap_log10 is not None else {}
    fig, axes = _axes_row(len(ch_names))
    for ci, ch in enumerate(ch_names):
        ax = axes[ci]
        for stage, (mean, _, n) in curves.items():
            col = colors.get(stage, '#888888')
            lab = f'{stage} (n={n})'
            if res is not None:
                exps = res['exponent'][masks[stage], ci]
                if np.isfinite(exps).any():
                    lab += f'  exp={np.nanmean(exps):.2f}'
            ax.plot(freqs, mean[ci], color=col, label=lab)
            if stage in ap_curves:
                ax.plot(freqs, ap_curves[stage][0][ci], color=col, ls='--', lw=1.0, alpha=0.8)
        ax.axvspan(fit_range[0], fit_range[1], color='#000000', alpha=0.04)
        ax.set_xscale('log')
        ax.set_xticks([0.5, 1, 2, 4, 8, 16, 30, 45])
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_xlim(freqs[0], freqs[-1])
        ax.set_title(ch)
        ax.set_xlabel('Frequency (Hz)')
        ax.grid(alpha=0.2, which='both')
        if ci == 0:
            ax.set_ylabel('PSD (dB/Hz)')
            ax.legend(fontsize=8)
    fig.suptitle('log-log PSD per stage with the aperiodic fit (dashed); '
                 f'shaded = fit range {fit_range[0]:g}-{fit_range[1]:g} Hz')
    fig.tight_layout()
    return fig


# --- 5. aperiodic parameters across the night --------------------------------
def plot_aperiodic_over_night(meta, res, ch_names, custom_stages):
    """Exponent and offset of every epoch against its position in the night, coloured by
    stage - the quickest way to see a drift, an outlier stretch or a stage effect."""
    _, colors, _, _ = stage_style(custom_stages)
    epoch_idx = meta['epoch_idx'].values
    stages = meta['stage'].astype(str).values
    n_ch = len(ch_names)
    fig, axes = plt.subplots(2, max(n_ch, 1), figsize=(_fig_width(n_ch), 6.0),
                             sharex=True, squeeze=False)
    for ci, ch in enumerate(ch_names):
        for row, key, lab in [(0, 'exponent', 'Exponent'), (1, 'offset', 'Offset')]:
            ax = axes[row][ci]
            vals = res[key][:, ci]
            for stage in dict.fromkeys(stages):
                m = stages == stage
                ax.scatter(epoch_idx[m], vals[m], s=5, alpha=0.6,
                           color=colors.get(stage, '#888888'),
                           label=stage if (ci == 0 and row == 0) else None)
            ax.grid(alpha=0.2)
            if ci == 0:
                ax.set_ylabel(lab)
            if row == 0:
                ax.set_title(ch)
            else:
                ax.set_xlabel('Epoch index')
        axes[0][ci].set_ylim(*np.nanpercentile(res['exponent'], [0.5, 99.5])
                             if np.isfinite(res['exponent']).any() else (0, 1))
    if n_ch:
        axes[0][0].legend(fontsize=8, markerscale=2)
    fig.suptitle('Aperiodic parameters across the night (one point per epoch)')
    fig.tight_layout()
    return fig


# --- 6. band power per stage -------------------------------------------------
def plot_band_power(band_vals, ch_names, masks, custom_stages, measure, measure_label):
    """Boxplot of the per-epoch band power, grouped by band and coloured by stage."""
    _, colors, _, _ = stage_style(custom_stages)
    bands = list(band_vals)
    stages = [s for s in masks if masks[s].sum() > 0]
    fig, axes = _axes_row(len(ch_names), height=4.6, sharey=True)
    width = 0.8 / max(len(stages), 1)
    for ci, ch in enumerate(ch_names):
        ax = axes[ci]
        for si, stage in enumerate(stages):
            data, pos = [], []
            for bi, band in enumerate(bands):
                v = band_vals[band].get(measure)
                if v is None:
                    continue
                vals = v[masks[stage], ci]
                vals = vals[np.isfinite(vals)]
                if len(vals) == 0:
                    continue
                data.append(vals)
                pos.append(bi + (si - (len(stages) - 1) / 2) * width)
            if not data:
                continue
            bp = ax.boxplot(data, positions=pos, widths=width * 0.85, patch_artist=True,
                            showfliers=False, medianprops=dict(color='black', lw=0.8))
            for patch in bp['boxes']:
                patch.set_facecolor(colors.get(stage, '#888888'))
                patch.set_alpha(0.75)
                patch.set_linewidth(0.5)
        ax.set_xticks(range(len(bands)))
        ax.set_xticklabels(bands, rotation=30, ha='right')
        ax.set_title(ch)
        ax.grid(alpha=0.2, axis='y')
        if ci == 0:
            ax.set_ylabel(measure_label)
    handles = [matplotlib.patches.Patch(facecolor=colors.get(s, '#888888'), alpha=0.75, label=s)
               for s in stages]
    if len(axes):
        axes[0].legend(handles=handles, fontsize=8, ncol=2)
    fig.suptitle(f'Band power per stage - {measure_label}')
    fig.tight_layout()
    return fig


# --- HTML tables -------------------------------------------------------------
def params_html(cfg):
    """Two-column 'parameter -> value' table of everything the run used."""
    rows = ''.join(f'<tr><td style="padding:2px 12px 2px 0"><b>{k}</b></td>'
                   f'<td>{v}</td></tr>' for k, v in cfg.items())
    return f'<table style="font-size:90%">{rows}</table>'


def coverage_html(agg, ch_names):
    """n_epochs (and n_fit_ok) per stage x channel - shows where `min epochs` produced NaN."""
    if agg is None or len(agg) == 0:
        return '<p><i>(no data)</i></p>'
    piv = agg.drop_duplicates(['stage', 'channel'])[['stage', 'channel', 'n_epochs', 'n_fit_ok']]
    wide = piv.pivot(index='stage', columns='channel', values='n_epochs').reindex(columns=ch_names)
    return wide.reset_index().to_html(index=False, border=0, na_rep='')


def guidance_html(cfg, data_dir, reports_dir, has_subject_info=False):
    """"Where do I look?" - the closing section of the database report.

    The run writes seven kinds of table; without a map it is easy to open the wrong one. This
    names the two files that answer most questions, then lists the rest with what each is for."""
    space = 'from_log' if 'log' in cfg['spaces'] else 'from_lin'
    band_col = (f'power_db_{space}_mean' if 'power_db' in cfg['measures']
                else (f'{cfg["measures"][0]}_mean' if cfg['measures'] else 'power_db_mean'))
    rows = [
        ('<b>Band power per participant</b>, ready for statistics',
         '<code>global_spectral_stage.tsv</code>', 'one row per participant x stage x channel x band; '
         f'the main column is <code>{band_col}</code>'),
        ('<b>Aperiodic exponent and offset</b> per participant',
         '<code>global_aperiodic_stage.tsv</code>',
         '<code>exponent_mean</code> and <code>offset_mean</code>, one row per participant x stage '
         'x channel'),
        ('Everything above in one file', '<code>spectral_features_database.xlsx</code>',
         'the same tables as Excel sheets, plus <code>coverage</code> and <code>parameters</code>'),
        ('Redraw a spectrum without recomputing', '<code>global_psd_stage.tsv</code>',
         'the mean PSD per stage x channel x frequency bin'),
        ('Look inside one participant', '<code>{file_id}_spectral_report.html</code>',
         'the per-participant figures and the parameters actually used'),
        ('Work at the epoch level (time course, outliers)',
         '<code>{file_id}_bandpower_epoch.tsv</code> / <code>{file_id}_aperiodic_epoch.tsv</code>',
         'one row per epoch x channel (x band); this is where a night-long time course lives'),
        ('Individual oscillatory peaks', '<code>{file_id}_periodic_peaks.tsv</code>',
         'centre frequency, power and bandwidth of every peak specparam fitted'),
    ]
    redraw = (
        '<p><small>Every figure of these reports can be redrawn from the tables, so you are never '
        'forced to re-run the tool to change a grouping:</small></p>'
        '<table style="font-size:90%">'
        '<tr><td style="padding:2px 14px 2px 0">mean PSD per stage (with its +/- SEM band)</td>'
        '<td><code>global_psd_stage.tsv</code> &mdash; <code>sem = psd_*_sd / sqrt(n_epochs)</code></td></tr>'
        '<tr><td style="padding:2px 14px 2px 0">PSD with the aperiodic component removed</td>'
        '<td><code>global_psd_stage.tsv</code> &mdash; <code>psd_ap_removed_db</code></td></tr>'
        '<tr><td style="padding:2px 14px 2px 0">log-log PSD + the aperiodic fit</td>'
        '<td>the same table, plus <code>offset_mean</code> / <code>exponent_mean</code> from '
        '<code>global_aperiodic_stage.tsv</code>: '
        '<code>ap_dB(f) = 10*(offset - exponent*log10(f))</code></td></tr>'
        '<tr><td style="padding:2px 14px 2px 0">exponent / offset across the night</td>'
        '<td><code>{file_id}_aperiodic_epoch.tsv</code></td></tr>'
        '<tr><td style="padding:2px 14px 2px 0">band-power boxplots, clean-epoch hypnogram</td>'
        '<td><code>{file_id}_bandpower_epoch.tsv</code> (it carries <code>epoch_idx</code> and '
        '<code>stage</code>)</td></tr></table>')
    joined = ('The participant-info table you selected is <b>already joined</b> into the global '
              'tables, so its columns can be used directly as grouping keys.'
              if has_subject_info else
              'No participant-info table was selected; to group by population variables, merge your '
              'own table on <code>file_id</code> (or re-run with the <i>Participant info</i> '
              'chooser filled in).')
    regroup = (
        f'<p><small>{joined}</small></p>'
        '<pre style="font-size:85%;background:#f6f6f6;padding:8px">'
        "import pandas as pd\n"
        "df = pd.read_csv('global_spectral_stage.tsv', sep='\\t', dtype={'file_id': str})\n"
        "# df = df.merge(pd.read_csv('my_population.csv'), on='file_id')   # if not joined already\n"
        "sel = df[(df.stage == 'N2') &amp; (df.band == 'sigma') &amp; (df.n_epochs >= 20)]\n"
        "sel.groupby(['group', 'channel'])['power_db_from_log_mean'].agg(['mean', 'sem', 'count'])"
        '</pre>')
    body = ''.join(f'<tr><td style="padding:3px 14px 3px 0;vertical-align:top">{q}</td>'
                   f'<td style="padding:3px 14px 3px 0;vertical-align:top">{f}</td>'
                   f'<td style="padding:3px 0;vertical-align:top"><small>{d}</small></td></tr>'
                   for q, f, d in rows)
    caveats = []
    if 'log' in cfg['spaces'] and 'lin' in cfg['spaces']:
        caveats.append('Both averaging spaces were exported: <code>_from_log</code> (mean of the '
                       'per-epoch dB) and <code>_from_lin</code> (dB of the linear mean). Pick one '
                       'and stay with it - mixing them across analyses is not comparable.')
    caveats.append('A cell computed on fewer than '
                   f'{cfg["min_epochs"]} epochs is <b>NaN</b>, but its <code>n_epochs</code> is '
                   'kept: always check that column before reading a value.')
    caveats.append('Rows with <code>n_epochs = 0</code> are channels a participant does not have '
                   '(dropped by tool 7/7bis); they are there so the table stays rectangular.')
    if any(m in cfg['measures'] for m in ('power_ap_removed_db', 'power_ratio_over_aperiodic',
                                          'power_rel_periodic')):
        caveats.append('The 1/f-corrected measures rest on the specparam fit, which is extrapolated '
                       'below its lower bound - read <b>delta</b> with care and check the '
                       '<code>n_fit_ok</code> column.')
    return (f'<p>Data tables: <code>{data_dir}</code><br>'
            f'Reports and database tables: <code>{reports_dir}</code></p>'
            f'<table style="font-size:95%">{body}</table>'
            '<p style="margin-top:14px"><b>Before you read a number</b></p><ul>'
            + ''.join(f'<li><small>{c}</small></li>' for c in caveats) + '</ul>'
            + '<p style="margin-top:14px"><b>Redrawing a figure yourself</b></p>' + redraw
            + '<p style="margin-top:14px"><b>Grouping participants differently</b></p>' + regroup)


def save_report_html(path, title, items):
    """Write an mne.Report from an ORDERED list of ('fig'|'html', title, payload) items.

    Ordered rather than "all figures then all text" so a figure can be placed after the section it
    belongs with. `None` payloads are skipped, which keeps the call sites free of conditionals."""
    report = mne.Report(title=title, verbose=False)
    for kind, item_title, payload in items:
        if payload is None:
            continue
        if kind == 'fig':
            report.add_figure(fig=payload, title=item_title, image_format='PNG')
        else:
            report.add_html(html=payload, title=item_title)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    report.save(str(path), overwrite=True, open_browser=False, verbose=False)
    return path


def close_figs(items):
    """Close every figure of an item list (matplotlib keeps them open otherwise)."""
    for kind, _, payload in items:
        if kind == 'fig' and payload is not None:
            plt.close(payload)


# --- database-level figures --------------------------------------------------
def pick_psd_column(psd_df, prefer=('psd_db_from_log', 'psd_db_from_lin')):
    """First available dB column of the global PSD table (which one exists depends on the
    averaging space the user selected)."""
    if psd_df is None or len(psd_df) == 0:
        return None
    for col in prefer:
        if col in psd_df.columns and psd_df[col].notna().any():
            return col
    return None


def plot_group_psd(psd_df, custom_stages, value_col=None,
                   ylabel='PSD (dB/Hz)', title='Group mean PSD per stage'):
    """Mean across participants (+/- SEM between subjects) of the per-stage mean PSD."""
    if psd_df is None or len(psd_df) == 0:
        return None
    value_col = value_col or pick_psd_column(psd_df)
    if value_col is None or value_col not in psd_df.columns:
        return None
    _, colors, _, _ = stage_style(custom_stages)
    ch_names = sorted(psd_df['channel'].unique())
    fig, axes = _axes_row(len(ch_names), height=4.4)
    for ci, ch in enumerate(ch_names):
        ax = axes[ci]
        sub_ch = psd_df[psd_df['channel'] == ch]
        for stage in [s for s in AASM_STAGES + list(custom_stages)
                      if s in set(sub_ch['stage'])]:
            sub = sub_ch[sub_ch['stage'] == stage]
            piv = sub.pivot_table(index='freq_hz', columns='file_id', values=value_col)
            if piv.empty:
                continue
            freqs = piv.index.values
            mean = np.nanmean(piv.values, axis=1)
            n = np.sum(np.isfinite(piv.values), axis=1)
            sem = np.nanstd(piv.values, axis=1, ddof=1) / np.sqrt(np.maximum(n, 1))
            col = colors.get(stage, '#888888')
            ax.plot(freqs, mean, color=col, label=f'{stage} (n={int(np.nanmax(n))})')
            ax.fill_between(freqs, mean - sem, mean + sem, color=col, alpha=0.25)
        ax.set_title(ch)
        ax.set_xlabel('Frequency (Hz)')
        ax.grid(alpha=0.2)
        if ci == 0:
            ax.set_ylabel(ylabel)
            ax.legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def plot_group_band(stage_df, column, label, custom_stages):
    """Distribution across participants of one aggregated band measure, per stage and band."""
    if stage_df is None or len(stage_df) == 0 or column not in stage_df.columns:
        return None
    _, colors, _, _ = stage_style(custom_stages)
    ch_names = sorted(stage_df['channel'].unique())
    bands = list(dict.fromkeys(stage_df['band']))
    stages = [s for s in AASM_STAGES + list(custom_stages) if s in set(stage_df['stage'])]
    fig, axes = _axes_row(len(ch_names), height=4.6)
    width = 0.8 / max(len(stages), 1)
    for ci, ch in enumerate(ch_names):
        ax = axes[ci]
        sub_ch = stage_df[stage_df['channel'] == ch]
        for si, stage in enumerate(stages):
            data, pos = [], []
            for bi, band in enumerate(bands):
                v = sub_ch[(sub_ch['stage'] == stage) & (sub_ch['band'] == band)][column].values
                v = v[np.isfinite(v)]
                if len(v) == 0:
                    continue
                data.append(v)
                pos.append(bi + (si - (len(stages) - 1) / 2) * width)
            if not data:
                continue
            bp = ax.boxplot(data, positions=pos, widths=width * 0.85, patch_artist=True,
                            showfliers=False, medianprops=dict(color='black', lw=0.8))
            for patch in bp['boxes']:
                patch.set_facecolor(colors.get(stage, '#888888'))
                patch.set_alpha(0.75)
                patch.set_linewidth(0.5)
        ax.set_xticks(range(len(bands)))
        ax.set_xticklabels(bands, rotation=30, ha='right')
        ax.set_title(ch)
        ax.grid(alpha=0.2, axis='y')
        if ci == 0:
            ax.set_ylabel(label)
    handles = [matplotlib.patches.Patch(facecolor=colors.get(s, '#888888'), alpha=0.75, label=s)
               for s in stages]
    if len(axes):
        axes[0].legend(handles=handles, fontsize=8, ncol=2)
    fig.suptitle(f'{label} across participants (one point per participant)')
    fig.tight_layout()
    return fig


def plot_group_aperiodic(ap_df, custom_stages):
    """Exponent and offset per stage across participants (one point per participant)."""
    if ap_df is None or len(ap_df) == 0:
        return None
    _, colors, _, _ = stage_style(custom_stages)
    ch_names = sorted(ap_df['channel'].unique())
    stages = [s for s in AASM_STAGES + list(custom_stages) if s in set(ap_df['stage'])]
    fig, axes = plt.subplots(2, max(len(ch_names), 1), figsize=(_fig_width(len(ch_names)), 6.4),
                             squeeze=False)
    for ci, ch in enumerate(ch_names):
        sub_ch = ap_df[ap_df['channel'] == ch]
        for row, col_name, lab in [(0, 'exponent_mean', 'Exponent'), (1, 'offset_mean', 'Offset')]:
            ax = axes[row][ci]
            data, keep = [], []
            for stage in stages:
                v = sub_ch[sub_ch['stage'] == stage][col_name].values
                v = v[np.isfinite(v)]
                if len(v):
                    data.append(v)
                    keep.append(stage)
            if data:
                bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                                medianprops=dict(color='black', lw=0.8))
                for patch, stage in zip(bp['boxes'], keep):
                    patch.set_facecolor(colors.get(stage, '#888888'))
                    patch.set_alpha(0.75)
                ax.set_xticks(range(1, len(keep) + 1))
                ax.set_xticklabels(keep)
            ax.grid(alpha=0.2, axis='y')
            if ci == 0:
                ax.set_ylabel(lab)
            if row == 0:
                ax.set_title(ch)
    fig.suptitle('Aperiodic parameters per stage across participants')
    fig.tight_layout()
    return fig


def plot_group_counts(stage_df):
    """Clean epochs per participant, split by stage - the sample size behind every value."""
    if stage_df is None or len(stage_df) == 0:
        return None
    sub = stage_df[stage_df['stage'].isin(AASM_STAGES)]
    if len(sub) == 0:
        return None
    counts = (sub.drop_duplicates(['file_id', 'stage', 'channel'])
              .groupby(['file_id', 'stage'])['n_epochs'].max().unstack(fill_value=0))
    counts = counts.reindex(columns=[s for s in AASM_STAGES if s in counts.columns])
    fig, ax = plt.subplots(figsize=(max(6.0, 0.45 * len(counts) + 3), 4.0))
    bottom = np.zeros(len(counts))
    for stage in counts.columns:
        vals = counts[stage].values.astype(float)
        ax.bar(range(len(counts)), vals, bottom=bottom,
               color=BASE_STAGE_COLORS.get(stage, '#888888'), label=stage)
        bottom += vals
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=90, fontsize=7)
    ax.set_ylabel('clean epochs')
    ax.set_title('Clean epochs per participant and stage')
    ax.legend(fontsize=8, ncol=5)
    fig.tight_layout()
    return fig

## 1 — Folders and scan

Point the tool at the folder holding the cleaned epochs (`*_clean-epo.fif` from tool 7 or 7bis). If none
is found there, tool 6's un-rejected `*_all-epo.fif` are used instead, with a warning.

The **Data folder** is optional: selecting it just pre-points the chooser below and picks up
`config_param/custom_stages.json`. The **Participant info** table is optional too; when given, its rows
are joined to the aggregated outputs on the column you choose.

**Scan** lists the participants, reports how many are already processed, and shows which channels each
one carries — a channel dropped by tool 7/7bis is simply absent from that participant.

In [ ]:
# =============================================================================
# Section 1 - folders, scan, provenance
# =============================================================================
S = {}          # shared state across sections

fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Data folder</b> (holds <code>derivatives/</code>) &mdash; optional, '
                 'pre-points the folders below:')

fc_clean = FileChooser(os.getcwd())
fc_clean.show_only_dirs = True
fc_clean.title = ('<b>Clean-epochs folder</b> (holds <code>*_clean-epo.fif</code> from tool 7 or '
                  '7bis, e.g. <code>derivatives/clean_epo_auto</code>):')

fc_subj = FileChooser(os.getcwd())
fc_subj.filter_pattern = ['*.csv', '*.tsv']
fc_subj.title = ('<b>Participant info</b> (optional <code>.csv</code>/<code>.tsv</code> joined to the '
                 'aggregated outputs):')

dd_join_col = widgets.Dropdown(description='Join on:', options=[], value=None,
                               layout=widgets.Layout(width='320px', display='none'),
                               style={'description_width': 'initial'})
txt_custom = widgets.Text(description='Custom stages:', value='',
                          placeholder='comma-separated non-AASM labels, e.g. N4, MT',
                          layout=widgets.Layout(width='520px'),
                          style={'description_width': 'initial'})
cb_exclude_flagged = widgets.Checkbox(
    value=True, description='Exclude the epochs tool 6 flagged (raw-epochs source only)',
    indent=False, layout=widgets.Layout(width='560px', display='none'))

btn_scan = widgets.Button(description='Scan', button_style='primary', icon='search')
lbl_scan = widgets.HTML('<i>Pick the clean-epochs folder, then Scan.</i>')
out_scan = widgets.Output()


def _on_data(chooser):
    """Convenience: point the clean-epochs chooser into <data>/derivatives."""
    try:
        data = fc_data.selected_path
        if not data:
            return
        data = Path(data)
        start = data / 'derivatives' if (data / 'derivatives').is_dir() else data
        fc_clean.reset(path=str(start))
        fc_clean.title = ('<b>Clean-epochs folder</b> (holds <code>*_clean-epo.fif</code> from tool 7 '
                          'or 7bis, e.g. <code>derivatives/clean_epo_auto</code>):')
        fc_subj.reset(path=str(data))
        fc_subj.title = ('<b>Participant info</b> (optional <code>.csv</code>/<code>.tsv</code> joined '
                         'to the aggregated outputs):')
        cs = load_custom_stages(data)
        if cs:
            txt_custom.value = ', '.join(cs)
        lbl_scan.value = '<i>Data folder set &mdash; pick the clean-epochs folder, then Scan.</i>'
    except Exception as exc:
        lbl_scan.value = f'<span style="color:#c62828">Data folder error: {exc}</span>'


def _on_subj(chooser):
    """Load the optional participant table and offer its columns as the join key."""
    try:
        df, err = load_subject_info(fc_subj.selected)
        if df is None:
            dd_join_col.layout.display = 'none'
            lbl_scan.value = f'<span style="color:#ef6c00">Participant info unreadable: {err}</span>'
            return
        S['subj_info'] = df
        dd_join_col.options = list(df.columns)
        guess = [c for c in df.columns if norm(c) in ('file_id', 'fileid', 'sub_id', 'subid', 'id')]
        dd_join_col.value = guess[0] if guess else list(df.columns)[0]
        dd_join_col.layout.display = ''
        lbl_scan.value = (f'<i>Participant info loaded ({len(df)} rows) &mdash; check the join '
                          f'column, then Scan.</i>')
    except Exception as exc:
        lbl_scan.value = f'<span style="color:#c62828">Participant info error: {exc}</span>'


def scan_folder(_=None):
    """Discover the participants, read each file's header + metadata (no signal), locate tool 6's
    sidecar, and build the channel-coverage preview. Everything here is non-fatal."""
    with out_scan:
        clear_output()
        try:
            clean = fc_clean.selected_path
            if not clean:
                lbl_scan.value = '<span style="color:#c62828">Select the clean-epochs folder first.</span>'
                return
            clean_root = Path(clean)
            data_root = Path(fc_data.selected_path) if fc_data.selected_path else None
            if data_root is None:                      # derive it from the derivatives/ ancestor
                for anc in clean_root.parents:
                    if norm(anc.name) == norm('derivatives'):
                        data_root = anc.parent
                        break
            parts, suffix = find_participants(clean_root)
            if not parts:
                lbl_scan.value = ('<span style="color:#c62828">No <code>*_clean-epo.fif</code> nor '
                                  '<code>*_all-epo.fif</code> found in that folder.</span>')
                return
            source = detect_source(clean_root, suffix)
            cb_exclude_flagged.layout.display = '' if source == 'raw' else 'none'

            custom = parse_custom_field(txt_custom.value) or load_custom_stages(data_root)
            txt_custom.value = ', '.join(custom)

            rows, warn, stages_seen, ch_counts = [], [], set(), {}
            data_dir = clean_root.parent / DATA_DIRNAME
            reports_dir = (data_root / REPORTS_DIRNAME) if data_root else (clean_root.parent / REPORTS_DIRNAME)
            n_done = 0
            for p in parts:
                fid = p['file_id']
                try:
                    ep = mne.read_epochs(p['fif'], preload=False, verbose=False)
                    meta = ep.metadata
                    stages = sorted(set(meta['stage'].astype(str))) if meta is not None else []
                    n_ep, chs, sf = len(ep), list(ep.ch_names), float(ep.info['sfreq'])
                    epoch_len = len(ep.times) / sf
                    del ep
                except Exception as exc:
                    warn.append(f'{fid}: unreadable ({exc})')
                    continue
                stages_seen.update(stages)
                for ch in chs:
                    ch_counts[ch] = ch_counts.get(ch, 0) + 1
                side = find_sidecar(fid, p['folder'], clean_root, data_root)
                info = load_sidecar(side)
                if not info['found']:
                    warn.append(f'{fid}: no tool-6 <code>_preprocessing_params.json</code> found '
                                f'(provenance only; epoch length read from the .fif)')
                p.update({'n_epochs': n_ep, 'ch_names': chs, 'sfreq': sf,
                          'epoch_length_s': info['epoch_length_s'] or round(epoch_len, 3),
                          'stages': stages, 'sidecar': side, 'sidecar_info': info,
                          'out_data': data_dir / p['subtree'],
                          'out_reports': reports_dir / p['subtree']})
                done = ((p['out_data'] / f'{fid}_spectral_stage.tsv').exists()
                        and (p['out_reports'] / f'{fid}_spectral_report.html').exists())
                half = ((p['out_data'] / f'{fid}_spectral_stage.tsv').exists()
                        or (p['out_reports'] / f'{fid}_spectral_report.html').exists())
                p['done'] = done
                if done:
                    n_done += 1
                elif half:
                    warn.append(f'{fid}: data/report mismatch (interrupted run) &mdash; will be reprocessed')
                rows.append({'file_id': fid, 'subtree': str(p['subtree']), 'epochs': n_ep,
                             'channels': len(chs), 'sfreq': sf, 'epoch_s': p['epoch_length_s'],
                             'stages': ' '.join(stages), 'sidecar': 'yes' if info['found'] else 'NO',
                             'processed': 'yes' if done else ''})

            parts = [p for p in parts if 'n_epochs' in p]
            S.update({'parts': parts, 'suffix': suffix, 'source': source, 'clean_root': clean_root,
                      'data_root': data_root, 'data_dir': data_dir, 'reports_dir': reports_dir,
                      'custom_stages': custom, 'ch_union': sorted(ch_counts),
                      'ch_counts': ch_counts, 'stages_seen': stages_seen})

            # Windows refuses paths over 260 characters unless long paths are enabled; a deep
            # OneDrive tree plus these file names gets close, so say it before the run, not after.
            longest = max((len(str(p['out_data'] / f'{p["file_id"]}_spectral_stage_third.tsv'))
                           for p in parts), default=0)
            if longest > 245:
                warn.append(f'the longest output path is {longest} characters &mdash; close to the '
                            f'260-character Windows limit. Some files may fail to be written; move '
                            f'the dataset to a shorter path if that happens.')

            sfs = {p['sfreq'] for p in parts}
            lens = {p['epoch_length_s'] for p in parts}
            if len(sfs) > 1:
                warn.append(f'mixed sampling rates across participants: {sorted(sfs)}')
            if len(lens) > 1:
                warn.append(f'mixed epoch lengths across participants: {sorted(lens)}')

            display(HTML(f'<b>{len(parts)} participants</b> &mdash; source: <code>{source}</code> '
                         f'(<code>*{suffix}</code>) &nbsp;|&nbsp; <b>{n_done} / {len(parts)}</b> '
                         f'already processed'))
            if source == 'raw':
                display(HTML('<span style="color:#ef6c00"><b>&#9888; raw (un-rejected) epochs</b> '
                             '&mdash; no manual or automatic rejection was applied to this folder. '
                             'Tick the box above to at least drop what tool 6 flagged.</span>'))
            display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))
            cov = pd.DataFrame({'channel': S['ch_union'],
                                'participants': [ch_counts[c] for c in S['ch_union']]})
            cov['coverage'] = (100.0 * cov['participants'] / max(len(parts), 1)).round(0)
            display(HTML('<b>Channel coverage</b> (a channel dropped by tool 7/7bis is simply absent '
                         'from that participant; the aggregated tables keep it as NaN)'
                         + cov.to_html(index=False, border=0)))
            for w in warn:
                display(HTML(f'<span style="color:#ef6c00">&#9888; {w}</span>'))

            build_stage_checkboxes(sorted(stages_seen), custom)
            build_participant_list()
            lbl_scan.value = (f'<span style="color:#2e7d32">Scan done &mdash; {len(parts)} participants, '
                              f'{len(S["ch_union"])} distinct channels.</span>')
        except Exception as exc:
            lbl_scan.value = f'<span style="color:#c62828">Scan failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


fc_data.register_callback(_on_data)
fc_subj.register_callback(_on_subj)
btn_scan.on_click(scan_folder)

display(widgets.VBox([fc_data, fc_clean, fc_subj, dd_join_col, txt_custom, cb_exclude_flagged,
                      widgets.HBox([btn_scan, lbl_scan]), out_scan]))

## 2 — Parameters

Every value below is written to the report and to the `parameters` sheet of the workbook, so a result can
always be traced back to the settings that produced it.

In [ ]:
# =============================================================================
# Section 2 - parameters (PSD, aperiodic fit, bands, measures, stages)
# =============================================================================
_LBL = {'description_width': 'initial'}


def _w(px):
    """A FRESH width Layout. Never share one Layout object between widgets: ipywidgets keeps the
    reference, so hiding one widget through `layout.display` would hide every widget sharing it."""
    return widgets.Layout(width=f'{px}px')


def _ind(px=14):
    """A FRESH indent Layout (same rule as _w: one Layout instance per widget)."""
    return widgets.Layout(margin=f'0 0 0 {px}px')


def _desc(text, indent=14):
    """Grey explanatory line, at the same indent as the widgets it introduces."""
    return widgets.HTML(f'<div style="line-height:1.2;margin-left:{indent}px">'
                        f'<small style="color:#555">{text}</small></div>')


def _head(text):
    return widgets.HTML(f'<b style="font-size:110%">{text}</b>')


def _reveal(checkbox, *boxes):
    """Show/hide widgets from `checkbox`; the initial display is derived from the CURRENT value
    (an observe handler only fires on a change, so a pre-ticked box would stay hidden)."""
    for box in boxes:
        box.layout.display = '' if checkbox.value else 'none'

    def _on(change, boxes=boxes):
        for box in boxes:
            box.layout.display = '' if change['new'] else 'none'

    checkbox.observe(_on, names='value')


# ---------------------------------------------------------------------------
# 2a. PSD
# ---------------------------------------------------------------------------
dd_psd_method = widgets.Dropdown(description='Method:', options=['welch', 'multitaper'],
                                 value=DEF_PSD['method'], layout=_w(300), style=_LBL)
txt_fmin = widgets.FloatText(description='fmin (Hz):', value=DEF_PSD['fmin'], layout=_w(210), style=_LBL)
txt_fmax = widgets.FloatText(description='fmax (Hz):', value=DEF_PSD['fmax'], layout=_w(210), style=_LBL)
txt_win_s = widgets.FloatText(description='Window (s):', value=DEF_PSD['win_s'], layout=_w(210), style=_LBL)
txt_overlap = widgets.FloatText(description='Overlap (%):', value=DEF_PSD['overlap_pct'],
                                layout=_w(210), style=_LBL)
dd_window = widgets.Dropdown(description='Window shape:', options=['hann', 'hamming', 'boxcar'],
                             value=DEF_PSD['window'], layout=_w(300), style=_LBL)
txt_bandwidth = widgets.FloatText(description='Bandwidth (Hz):', value=DEF_PSD['bandwidth'],
                                  layout=_w(210), style=_LBL)

box_welch = widgets.HBox([txt_win_s, txt_overlap, dd_window], layout=_ind(14))
box_mt = widgets.HBox([txt_bandwidth], layout=_ind(14))


def _on_psd_method(change):
    welch = change['new'] == 'welch'
    box_welch.layout.display = '' if welch else 'none'
    box_mt.layout.display = 'none' if welch else ''


dd_psd_method.observe(_on_psd_method, names='value')
box_welch.layout.display = '' if dd_psd_method.value == 'welch' else 'none'
box_mt.layout.display = 'none' if dd_psd_method.value == 'welch' else ''

box_psd = widgets.VBox([
    _head('Power spectral density'),
    _desc('Welch = sliding-window average (the sleep-research standard); multitaper = smoother, '
          'slower. A 4 s window gives a 0.25 Hz resolution, fine enough for ~2 cycles of the lowest '
          'frequency of interest. fmax is capped at the Nyquist frequency of each file.'),
    widgets.HBox([dd_psd_method, txt_fmin, txt_fmax], layout=_ind(14)),
    box_welch, box_mt,
])

# ---------------------------------------------------------------------------
# 2b. Aperiodic / periodic fit (specparam)
# ---------------------------------------------------------------------------
cb_do_fit = widgets.Checkbox(value=True, indent=False, layout=_w(620),
                             description='Fit the aperiodic (1/f) component with specparam')
txt_fit_fmin = widgets.FloatText(description='Fit fmin (Hz):', value=DEF_FIT['fit_fmin'],
                                 layout=_w(210), style=_LBL)
txt_fit_fmax = widgets.FloatText(description='Fit fmax (Hz):', value=DEF_FIT['fit_fmax'],
                                 layout=_w(210), style=_LBL)
dd_ap_mode = widgets.Dropdown(description='Aperiodic mode:', options=['fixed', 'knee'],
                              value=DEF_FIT['aperiodic_mode'], layout=_w(300), style=_LBL)
txt_pw_min = widgets.FloatText(description='Peak width min (Hz):', value=DEF_FIT['peak_width_min'],
                               layout=_w(300), style=_LBL)
txt_pw_max = widgets.FloatText(description='max (Hz):', value=DEF_FIT['peak_width_max'],
                               layout=_w(210), style=_LBL)
txt_min_peak_height = widgets.FloatText(description='Min peak height:', value=DEF_FIT['min_peak_height'],
                                        layout=_w(300), style=_LBL)
txt_max_n_peaks = widgets.IntText(description='Max peaks:', value=DEF_FIT['max_n_peaks'],
                                  layout=_w(210), style=_LBL)
txt_peak_thr = widgets.FloatText(description='Peak threshold (SD):', value=DEF_FIT['peak_threshold'],
                                 layout=_w(300), style=_LBL)
txt_r2_min = widgets.FloatText(description='Flag fit as poor if R2 <', value=DEF_FIT['r2_min'],
                               layout=_w(300), style=_LBL)
txt_mae_max = widgets.FloatText(description='or MAE >', value=DEF_FIT['mae_max'],
                                layout=_w(210), style=_LBL)
cb_exclude_bad_fit = widgets.Checkbox(value=False, indent=False, layout=_w(620),
                                      description='Exclude poorly-fitted epochs from the aggregated means')

cb_smooth = widgets.Checkbox(value=DEF_SMOOTH['enabled'], indent=False, layout=_w(400),
                             description='Smooth the spectrum before fitting')
# Same checkbox width on both sub-rows so the two value fields line up vertically.
cb_smooth_lowess = widgets.Checkbox(value=DEF_SMOOTH['lowess'], indent=False, layout=_w(200),
                                    description='LOWESS span (Hz):')
txt_lowess_span = widgets.FloatText(value=DEF_SMOOTH['lowess_span_hz'], layout=_w(90))
cb_smooth_median = widgets.Checkbox(value=DEF_SMOOTH['median'], indent=False, layout=_w(200),
                                    description='Median span (Hz):')
txt_median_span = widgets.FloatText(value=DEF_SMOOTH['median_span_hz'], layout=_w(90))
cb_parallel = widgets.Checkbox(value=True, indent=False, layout=_w(300),
                               description='Fit epochs in parallel')
txt_n_jobs = widgets.IntText(description='workers (-1 = all cores):', value=-1,
                             layout=_w(260), style=_LBL)

# Each filter on its own line, one indent deeper than the master checkbox.
row_smooth = widgets.HBox([cb_smooth], layout=_ind(14))
row_smooth_lowess = widgets.HBox([cb_smooth_lowess, txt_lowess_span], layout=_ind(44))
row_smooth_median = widgets.HBox([cb_smooth_median, txt_median_span], layout=_ind(44))
_reveal(cb_smooth, row_smooth_lowess, row_smooth_median)
_reveal(cb_smooth_lowess, txt_lowess_span)
_reveal(cb_smooth_median, txt_median_span)
row_parallel = widgets.HBox([cb_parallel, txt_n_jobs], layout=_ind(14))
_reveal(cb_parallel, txt_n_jobs)

box_fit_inner = widgets.VBox([
    _desc('Fit range, independent of the PSD range. Starting at 2 Hz keeps slow waves from dragging '
          'the slope (tool-6 convention).'),
    widgets.HBox([txt_fit_fmin, txt_fit_fmax], layout=_ind(14)),
    _desc('<b>fixed</b> = a straight line in log-log, the right choice for a 2-45 Hz sleep spectrum. '
          'Use <b>knee</b> only when the log-log spectrum visibly bends (typically when fitting well '
          'above 45 Hz): with no real bend the knee parameter runs away and the fit gets worse. '
          'Check it on the log-log figure of the individual report.'),
    widgets.HBox([dd_ap_mode], layout=_ind(14)),
    _desc('Peak model: the oscillatory peaks are modelled and removed before the aperiodic component '
          'is read. Min width must stay above 2x the frequency resolution; capping the number of '
          'peaks keeps a noisy spectrum from sending the fit into a runaway peak search.', 44),
    widgets.HBox([txt_pw_min, txt_pw_max], layout=_ind(44)),
    widgets.HBox([txt_min_peak_height, txt_max_n_peaks, txt_peak_thr], layout=_ind(44)),
    _desc('Fit quality: epochs outside these bounds are kept but marked <code>fit_ok = False</code> '
          'and counted in the report. Tool 6 already rejects on the same criteria, so this should '
          'stay rare.', 44),
    widgets.HBox([txt_r2_min, txt_mae_max], layout=_ind(44)),
    widgets.HBox([cb_exclude_bad_fit], layout=_ind(44)),
    _desc('Smoothing (running median, then LOWESS, as in tool 6) applies to the copy of the spectrum '
          'that feeds the fit ONLY - the band powers always use the raw PSD.'),
    row_smooth, row_smooth_lowess, row_smooth_median, row_parallel,
], layout=_ind(14))
_reveal(cb_do_fit, box_fit_inner)

box_fit = widgets.VBox([
    _head('Aperiodic and periodic components (specparam)'),
    _desc('Fitted per epoch and per channel. Unticking skips specparam entirely (much faster) and '
          'disables the three measures below that need the 1/f fit.'),
    cb_do_fit, box_fit_inner,
])

# ---------------------------------------------------------------------------
# 2c. Frequency bands
# ---------------------------------------------------------------------------
band_rows = widgets.VBox([])
btn_add_band = widgets.Button(description='Add band', icon='plus', layout=_w(140))
lbl_bands = widgets.HTML()


def _band_row(name, lo, hi):
    t_name = widgets.Text(value=name, layout=_w(140))
    t_lo = widgets.FloatText(value=lo, layout=_w(110))
    t_hi = widgets.FloatText(value=hi, layout=_w(110))
    btn_del = widgets.Button(icon='trash', layout=_w(42), tooltip='Remove')
    row = widgets.HBox([t_name, t_lo, t_hi, btn_del])
    row._fields = (t_name, t_lo, t_hi)
    btn_del.on_click(lambda b: setattr(band_rows, 'children',
                                       tuple(r for r in band_rows.children if r is not row)))
    return row


def _add_band(_=None, name='new', lo=0.0, hi=1.0):
    band_rows.children = tuple(band_rows.children) + (_band_row(name, lo, hi),)


btn_add_band.on_click(lambda b: _add_band())
for _n, _lo, _hi in DEFAULT_BANDS:
    _add_band(name=_n, lo=_lo, hi=_hi)


def get_bands():
    """Read the band rows; returns (bands, errors). Bands are half-open [fmin, fmax)."""
    bands, errors, seen = [], [], set()
    for row in band_rows.children:
        t_name, t_lo, t_hi = row._fields
        name = t_name.value.strip()
        lo, hi = float(t_lo.value), float(t_hi.value)
        if not name:
            continue
        if name in seen:
            errors.append(f'duplicate band name "{name}"')
            continue
        if lo >= hi:
            errors.append(f'band "{name}": fmin must be < fmax')
            continue
        seen.add(name)
        bands.append((name, lo, hi))
    if not bands:
        errors.append('no frequency band defined')
    return bands, errors


box_bands = widgets.VBox([
    _head('Frequency bands'),
    _desc('Half-open intervals [fmin, fmax): adjacent bands never share a frequency bin, so when the '
          'bands tile the PSD range the relative powers sum to 1. Overlapping bands are allowed.'),
    widgets.VBox([widgets.HTML('<b style="width:140px;display:inline-block">name</b>'
                               '<b style="width:110px;display:inline-block">fmin (Hz)</b>'
                               '<b style="width:110px;display:inline-block">fmax (Hz)</b>'),
                  band_rows, btn_add_band, lbl_bands], layout=_ind(14)),
])

# ---------------------------------------------------------------------------
# 2d. Measures to export
# ---------------------------------------------------------------------------
measure_boxes = {}
_measure_widgets = []
for _key, _label, _descr, _default, _needs_fit in MEASURES:
    cb = widgets.Checkbox(value=_default, indent=False, description=f'{_label}  ({_key})',
                          layout=_w(620))
    measure_boxes[_key] = cb
    _measure_widgets += [cb, _desc(_descr, 44)]

box_measures = widgets.VBox([
    _head('Measures to export'),
    _desc('Every ticked measure becomes a column of the epoch-level and aggregated tables.'),
    widgets.VBox(_measure_widgets, layout=_ind(14)),
])


def _sync_measure_availability(change=None):
    """The three 1/f-based measures are only available when the fit runs."""
    on = cb_do_fit.value
    for key, cb in measure_boxes.items():
        if MEASURE_NEEDS_FIT[key]:
            cb.disabled = not on


cb_do_fit.observe(_sync_measure_availability, names='value')
_sync_measure_availability()

# ---------------------------------------------------------------------------
# 2e. Stages, aggregation, coverage
# ---------------------------------------------------------------------------
stage_box = widgets.HBox([])
stage_checkboxes = {}
cb_avg_log = widgets.Checkbox(value=True, indent=False, layout=_w(620),
                              description='Average in log space (mean of the per-epoch dB)')
cb_avg_lin = widgets.Checkbox(value=False, indent=False, layout=_w(620),
                              description='Average in linear space, then convert to dB')
txt_min_epochs = widgets.IntText(description='Min epochs per cell:', value=DEF_MIN_EPOCHS,
                                 layout=_w(300), style=_LBL)
cb_thirds = widgets.Checkbox(value=False, indent=False, layout=_w(620),
                             description='Also split the night into thirds (draft)')


def build_stage_checkboxes(stages_present, custom_stages):
    """Called by the scan: one checkbox per stage actually present in the files, all ticked."""
    order = [s for s in AASM_STAGES + list(custom_stages) if s in stages_present]
    order += [s for s in stages_present if s not in order]
    stage_checkboxes.clear()
    for st in order:
        stage_checkboxes[st] = widgets.Checkbox(value=True, indent=False, description=st,
                                                layout=_w(110))
    stage_box.children = tuple(stage_checkboxes.values())


box_stages = widgets.VBox([
    _head('Stages and aggregation'),
    _desc('Stages are listed after the scan (only those present in the files).'),
    widgets.VBox([stage_box], layout=_ind(14)),
    _desc('Averaging the epochs of a stage: <b>log space</b> averages the per-epoch dB values (what a '
          'per-epoch statistical model sees, robust to one loud epoch); <b>linear space</b> averages '
          'the power then converts to dB (preserves total power, but a few high-power epochs '
          'dominate). Ticking both exports the two, suffixed <code>_from_log</code> / '
          '<code>_from_lin</code>.'),
    widgets.VBox([cb_avg_log, cb_avg_lin], layout=_ind(14)),
    _desc('A (stage x channel) cell computed on fewer epochs than this is written as NaN, with '
          'n_epochs kept, so a shortage is visible instead of silent.'),
    widgets.HBox([txt_min_epochs], layout=_ind(14)),
    _desc('Thirds: the sleep period (first to last non-W epoch) split into three equal spans of '
          'epoch index, written to a companion table. NREM-REM cycle detection is not attempted.'),
    widgets.VBox([cb_thirds], layout=_ind(14)),
])

# ---------------------------------------------------------------------------
# Collect everything the run needs
# ---------------------------------------------------------------------------
lbl_params = widgets.HTML()


def get_params():
    """Read every Section-2 widget into one config dict; returns (cfg, errors)."""
    errors = []
    bands, band_err = get_bands()
    errors += band_err
    psd = dict(method=dd_psd_method.value, fmin=float(txt_fmin.value), fmax=float(txt_fmax.value),
               win_s=float(txt_win_s.value), overlap_pct=float(txt_overlap.value),
               window=dd_window.value, bandwidth=float(txt_bandwidth.value))
    if psd['fmin'] <= 0 or psd['fmin'] >= psd['fmax']:
        errors.append('PSD range: 0 < fmin < fmax is required')
    if not 0 <= psd['overlap_pct'] < 100:
        errors.append('Welch overlap must be in [0, 100)')
    fit = dict(fit_fmin=float(txt_fit_fmin.value), fit_fmax=float(txt_fit_fmax.value),
               aperiodic_mode=dd_ap_mode.value, peak_width_min=float(txt_pw_min.value),
               peak_width_max=float(txt_pw_max.value),
               min_peak_height=float(txt_min_peak_height.value),
               max_n_peaks=int(txt_max_n_peaks.value), peak_threshold=float(txt_peak_thr.value),
               r2_min=float(txt_r2_min.value), mae_max=float(txt_mae_max.value))
    do_fit = bool(cb_do_fit.value)
    if do_fit:
        if not HAS_SPECPARAM:
            errors.append('specparam is not installed - untick the aperiodic fit or install it')
        if fit['fit_fmin'] >= fit['fit_fmax']:
            errors.append('1/f fit range: fmin must be < fmax')
        if fit['fit_fmax'] > psd['fmax'] or fit['fit_fmin'] < psd['fmin']:
            errors.append('the 1/f fit range must sit inside the PSD range')
    measures = [k for k, cb in measure_boxes.items()
                if cb.value and (do_fit or not MEASURE_NEEDS_FIT[k])]
    if not measures:
        errors.append('no measure selected')
    stages_sel = [s for s, cb in stage_checkboxes.items() if cb.value]
    if not stages_sel:
        errors.append('no sleep stage selected (run the scan first)')
    spaces = ([['log'] if cb_avg_log.value else []] + [['lin'] if cb_avg_lin.value else []])
    spaces = [s for group in spaces for s in group]
    if not spaces:
        errors.append('select at least one averaging space (log and/or linear)')
    # Unticking both filters is the same as unticking the master: `enabled` is set accordingly so
    # the report never claims a smoothing that did not happen.
    use_lowess, use_median = bool(cb_smooth_lowess.value), bool(cb_smooth_median.value)
    smooth = dict(enabled=bool(cb_smooth.value) and (use_lowess or use_median),
                  lowess=use_lowess, median=use_median,
                  median_span_hz=float(txt_median_span.value),
                  lowess_span_hz=float(txt_lowess_span.value))
    if smooth['enabled'] and smooth['lowess'] and not HAS_LOWESS:
        errors.append('statsmodels (LOWESS) is missing - untick the LOWESS filter')
    cfg = dict(psd=psd, fit=fit, do_fit=do_fit, smooth=smooth, bands=bands, measures=measures,
               stages=stages_sel, spaces=spaces, min_epochs=int(txt_min_epochs.value),
               thirds=bool(cb_thirds.value), exclude_bad_fit=bool(cb_exclude_bad_fit.value),
               n_jobs=int(txt_n_jobs.value) if cb_parallel.value else 1)
    return cfg, errors


display(widgets.VBox([box_psd, box_fit, box_bands, box_measures, box_stages, lbl_params]))

## 3 — Participants

All participants are ticked by default. *Skip already processed* leaves out the ones that already have
both their aggregated table and their report on disk; a participant with only one of the two (an
interrupted run) is always reprocessed.

In [ ]:
# =============================================================================
# Section 3 - participant selection
# =============================================================================
part_box = widgets.VBox([])
part_checkboxes = {}
cb_skip = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='420px'),
                           description='Skip already processed participants')
btn_all = widgets.Button(description='Select all', layout=widgets.Layout(width='120px'))
btn_none = widgets.Button(description='None', layout=widgets.Layout(width='90px'))
lbl_part = widgets.HTML('<i>Run the scan in Section 1 first.</i>')


def build_participant_list():
    """One checkbox per participant, ticked by default; the already-processed ones are annotated."""
    part_checkboxes.clear()
    rows = []
    for p in S.get('parts', []):
        tag = ' <span style="color:#2e7d32">(already processed)</span>' if p.get('done') else ''
        side = '' if p['sidecar_info']['found'] else ' <span style="color:#ef6c00">(no sidecar)</span>'
        cb = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='300px'),
                              description=p['file_id'])
        part_checkboxes[p['file_id']] = cb
        rows.append(widgets.HBox([cb, widgets.HTML(
            f'<small>{p["n_epochs"]} epochs &middot; {len(p["ch_names"])} ch &middot; '
            f'{p["sfreq"]:g} Hz &middot; {p["epoch_length_s"]:g} s{tag}{side}</small>')]))
    part_box.children = tuple(rows)
    n_done = sum(1 for p in S.get('parts', []) if p.get('done'))
    lbl_part.value = (f'<b>{len(part_checkboxes)}</b> participants &mdash; '
                      f'<b>{n_done} / {len(part_checkboxes)}</b> already processed.')


def selected_participants():
    """The participants to run, honouring the checkboxes and the skip option."""
    out = []
    for p in S.get('parts', []):
        cb = part_checkboxes.get(p['file_id'])
        if cb is None or not cb.value:
            continue
        if cb_skip.value and p.get('done'):
            continue
        out.append(p)
    return out


btn_all.on_click(lambda b: [setattr(cb, 'value', True) for cb in part_checkboxes.values()])
btn_none.on_click(lambda b: [setattr(cb, 'value', False) for cb in part_checkboxes.values()])

display(widgets.VBox([widgets.HBox([btn_all, btn_none, cb_skip]), lbl_part, part_box]))

## 4 — Run

Each participant is processed independently: a failure is logged and the run continues. Per participant
the tool writes the epoch-level tables, the aggregated tables and an HTML report; at the end it rebuilds
the database-level tables by re-reading every per-participant table from disk, and writes the Excel
workbook and the database report.

In [ ]:
# =============================================================================
# Section 4 - run: per participant, then the database-level outputs
# =============================================================================
EXCEL_MAX_ROWS = 1048575          # Excel's hard limit, minus the header row

btn_run = widgets.Button(description='Run', button_style='success', icon='play')
lbl_run = widgets.HTML()
progress_part = widgets.IntProgress(value=0, min=0, max=1, description='Participants:',
                                    layout=widgets.Layout(width='620px'),
                                    style={'description_width': 'initial'})
progress_step = widgets.IntProgress(value=0, min=0, max=1, bar_style='info',
                                    description='Pipeline:', layout=widgets.Layout(width='620px'),
                                    style={'description_width': 'initial'})
lbl_phase = widgets.HTML()
out_run = widgets.Output()

_STEP_LEGEND = widgets.HTML(
    '<div style="width:620px;font-size:85%;color:#555;display:flex;text-align:center">'
    + ''.join(f'<div style="flex:{c};border-right:1px solid #ccc">{n}</div>'
              for n, c in [('Load', COST_LOAD), ('PSD', COST_PSD), ('1/f fit', COST_FIT),
                           ('Bands', COST_BANDS), ('Report', COST_REPORT)])
    + '</div>')


def _write_tsv(df, path):
    """Write a table, creating the folder. Empty tables are skipped (nothing to say)."""
    if df is None or len(df) == 0:
        return None
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    return path


def _join_subject_info(df):
    """Left-join the optional participant table on file_id (normcase-insensitive on both sides)."""
    info = S.get('subj_info')
    col = dd_join_col.value
    if df is None or len(df) == 0 or info is None or not col or col not in info.columns:
        return df
    try:
        left = df.copy()
        left['_key'] = left['file_id'].astype(str).map(norm)
        right = info.copy()
        right['_key'] = right[col].astype(str).map(norm)
        right = right.drop_duplicates('_key')
        merged = left.merge(right.drop(columns=[c for c in [col] if c in right.columns]),
                            on='_key', how='left').drop(columns=['_key'])
        return merged
    except Exception:
        return df


def process_participant(p, cfg, log):
    """Full pipeline for one participant. Returns True on success; a fatal step raises."""
    fid = p['file_id']
    steps = COST_LOAD + COST_PSD + COST_BANDS + COST_REPORT + (COST_FIT if cfg['do_fit'] else 0)
    progress_step.max = steps
    progress_step.value = 0

    def phase(name, add):
        lbl_phase.value = f'<small>{fid} &mdash; {name}</small>'
        progress_step.value = min(progress_step.value + add, steps)

    # --- load ---------------------------------------------------------------
    phase('loading epochs', 0)
    epochs = mne.read_epochs(p['fif'], preload=True, verbose=False)
    meta = epochs.metadata.reset_index(drop=True).copy()
    if 'stage' not in meta.columns:
        raise ValueError('the .fif metadata has no `stage` column')
    if 'epoch_idx' not in meta.columns:
        meta['epoch_idx'] = np.arange(len(meta))
    if S['source'] == 'raw' and cb_exclude_flagged.value and 'reject_flag' in meta.columns:
        keep = ~meta['reject_flag'].astype(bool).values
        epochs = epochs[keep]
        meta = meta[keep].reset_index(drop=True)
        log(f'  dropped {int((~keep).sum())} epochs flagged by tool 6')
    if len(epochs) == 0:
        raise ValueError('no epoch left to analyse')
    stages = meta['stage'].astype(str).values
    ch_names = list(epochs.ch_names)
    phase('loading epochs', COST_LOAD)

    # --- PSD ----------------------------------------------------------------
    phase('computing the PSD', 0)
    freqs, psds, notes = compute_psd_array(epochs, **cfg['psd'])
    for n in notes:
        log(f'  &#9888; {n}')
    del epochs
    phase('computing the PSD', COST_PSD)

    # --- aperiodic fit ------------------------------------------------------
    res, peak_rows, ap_log10 = None, [], None
    if cfg['do_fit']:
        phase('fitting the aperiodic component', 0)
        psd_fit = smooth_for_fit(psds, freqs, cfg['smooth'])
        base = progress_step.value

        def _fit_progress(done, total):
            progress_step.value = min(base + int(COST_FIT * done / max(total, 1)), steps)

        res, peak_rows = fit_spectra(freqs, psd_fit, cfg['fit'], n_jobs=cfg['n_jobs'],
                                     progress=_fit_progress)
        del psd_fit
        ap_log10 = aperiodic_psd_log10(freqs, res, cfg['fit']['aperiodic_mode'])
        ok = float(np.nanmean(res['fit_ok'])) * 100.0
        log(f'  1/f fit: {ok:.0f} % of spectra within the quality bounds, '
            f'{len(peak_rows)} peaks detected')
        if cfg['fit']['aperiodic_mode'] == 'knee':
            med_knee = np.nanmedian(np.abs(res['knee']))
            if np.isfinite(med_knee) and med_knee > KNEE_DEGENERATE:
                log(f'  &#9888; the knee parameter is degenerate (median |knee| = {med_knee:.3g}): '
                    f'this spectrum has no real bend &mdash; use the <code>fixed</code> mode')
        progress_step.value = min(base + COST_FIT, steps)

    # --- band measures ------------------------------------------------------
    phase('extracting band power', 0)
    # A band needs at least two frequency bins to mean anything; one that is only partly inside the
    # PSD range is kept but flagged, since its value is not comparable to a fully covered band.
    df_hz = float(np.median(np.diff(freqs)))
    usable = []
    for n, lo, hi in cfg['bands']:
        n_bins = int(((freqs >= lo) & (freqs < hi)).sum())
        if n_bins < 2:
            log(f'  &#9888; band "{n}" ({lo:g}-{hi:g} Hz) has {n_bins} frequency bin(s) inside the '
                f'PSD range - skipped')
            continue
        span = min(hi, float(freqs[-1]) + df_hz) - max(lo, float(freqs[0]))
        if span < 0.99 * (hi - lo):
            log(f'  &#9888; band "{n}" ({lo:g}-{hi:g} Hz) is only {100 * span / (hi - lo):.0f} % '
                f'inside the PSD range - its power is not comparable to a fully covered band')
        usable.append((n, lo, hi))
    if not usable:
        raise ValueError('no frequency band fits inside the PSD range')
    band_vals = compute_band_measures(freqs, psds, ap_log10, usable, cfg['measures'])
    thirds = assign_thirds(meta['epoch_idx'].values, stages) if cfg['thirds'] else None

    stages_sel = [s for s in cfg['stages'] if s in set(stages)]
    masks_el = stage_masks(stages, stages_sel)
    fit_ok = res['fit_ok'] if res is not None else None

    agg = aggregate_bandpower(fid, S['source'], band_vals, ch_names, masks_el, cfg['measures'],
                              cfg['min_epochs'], spaces=cfg['spaces'], fit_ok=fit_ok,
                              exclude_bad_fit=cfg['exclude_bad_fit'])
    psd_stage = aggregate_psd(fid, S['source'], freqs, psds, ap_log10, ch_names, masks_el,
                              cfg['min_epochs'], spaces=cfg['spaces'])
    ap_stage = (aggregate_aperiodic(fid, S['source'], res, ch_names, masks_el, cfg['min_epochs'],
                                    exclude_bad_fit=cfg['exclude_bad_fit'])
                if res is not None else None)
    phase('extracting band power', COST_BANDS)

    # --- write the data BEFORE the report (interruption safety) -------------
    out_data, out_reports = p['out_data'], p['out_reports']
    _write_tsv(build_bandpower_epoch(fid, S['source'], meta, ch_names, band_vals, cfg['measures'],
                                     thirds), out_data / f'{fid}_bandpower_epoch.tsv')
    if res is not None:
        _write_tsv(build_aperiodic_epoch(fid, S['source'], meta, ch_names, res, thirds),
                   out_data / f'{fid}_aperiodic_epoch.tsv')
        _write_tsv(build_peaks_table(fid, S['source'], meta, ch_names, peak_rows),
                   out_data / f'{fid}_periodic_peaks.tsv')
        _write_tsv(ap_stage, out_data / f'{fid}_aperiodic_stage.tsv')
    _write_tsv(psd_stage, out_data / f'{fid}_psd_stage.tsv')
    if cfg['thirds']:
        masks_third = {f'{st}|{th}': (stages == st) & (thirds == th)
                       for st in stages_sel for th in ['T1', 'T2', 'T3']}
        third_agg = aggregate_bandpower(fid, S['source'], band_vals, ch_names, masks_third,
                                        cfg['measures'], cfg['min_epochs'], spaces=cfg['spaces'],
                                        fit_ok=fit_ok, exclude_bad_fit=cfg['exclude_bad_fit'])
        if len(third_agg):
            third_agg[['stage', 'third']] = third_agg['stage'].str.split('|', expand=True)
            _write_tsv(third_agg, out_data / f'{fid}_spectral_stage_third.tsv')
    _write_tsv(agg, out_data / f'{fid}_spectral_stage.tsv')      # written last = the skip marker

    # --- report -------------------------------------------------------------
    phase('building the report', 0)
    custom = S.get('custom_stages', [])
    psd_lin = np.where(psds > 0, psds, np.nan)
    psd_db = 10.0 * np.log10(psd_lin)
    use_log, use_lin = 'log' in cfg['spaces'], 'lin' in cfg['spaces']
    items = [('fig', 'Clean epochs', plot_clean_hypnogram(meta, custom, fid)),
             ('fig', 'Mean PSD per stage',
              plot_psd_per_stage(freqs, masks_el, ch_names, custom, cfg['min_epochs'],
                                 'Mean PSD per stage', 'PSD (dB/Hz)',
                                 values_db=psd_db if use_log else None,
                                 values_lin=psd_lin if use_lin else None))]
    if ap_log10 is not None:
        # In linear space the "1/f removed" spectrum is the PSD / aperiodic-fit ratio, whose dB is
        # exactly the log-space difference averaged the other way round.
        items.append(('fig', 'PSD after removing the aperiodic component',
                      plot_psd_per_stage(freqs, masks_el, ch_names, custom, cfg['min_epochs'],
                                         'Mean PSD per stage after removing the aperiodic component',
                                         'Power above the 1/f fit (dB)',
                                         values_db=(psd_db - 10.0 * ap_log10) if use_log else None,
                                         values_lin=(psd_lin / 10.0 ** ap_log10) if use_lin else None)))
        items.append(('fig', 'log-log PSD with the aperiodic fit',
                      plot_loglog_with_fit(freqs, psds, ap_log10, masks_el, ch_names, custom,
                                           cfg['min_epochs'],
                                           (cfg['fit']['fit_fmin'], cfg['fit']['fit_fmax']), res)))
        items.append(('fig', 'Aperiodic parameters across the night',
                      plot_aperiodic_over_night(meta, res, ch_names, custom)))
    band_measure = ('power_db' if 'power_db' in cfg['measures'] else cfg['measures'][0])
    band_label = dict((k, lbl) for k, lbl, _, _, _ in MEASURES)[band_measure]
    items.append(('fig', f'Band power per stage ({band_label})',
                  plot_band_power(band_vals, ch_names, masks_el, custom, band_measure, band_label)))

    side = p['sidecar_info']
    params = {
        'Source': f"{S['source']} ({p['fif'].name})",
        'Epochs analysed': f'{len(meta)} of {int(meta["epoch_idx"].max()) + 1} in the recording',
        'Channels': ', '.join(ch_names),
        'Sampling rate': f"{p['sfreq']:g} Hz",
        'Epoch length': f"{p['epoch_length_s']:g} s",
        'PSD': (f"{cfg['psd']['method']}, {cfg['psd']['fmin']:g}-{cfg['psd']['fmax']:g} Hz, "
                + (f"{cfg['psd']['win_s']:g} s {cfg['psd']['window']} window, "
                   f"{cfg['psd']['overlap_pct']:g} % overlap"
                   if cfg['psd']['method'] == 'welch'
                   else f"bandwidth {cfg['psd']['bandwidth']:g} Hz")
                + f", resolution {freqs[1] - freqs[0]:.3g} Hz"),
        'Bands': ', '.join(f'{n} {lo:g}-{hi:g}' for n, lo, hi in usable),
        'Measures': ', '.join(cfg['measures']),
        'Epoch averaging': ' + '.join({'log': 'log space (mean of the per-epoch dB)',
                                       'lin': 'linear space, then converted to dB'}[s]
                                      for s in cfg['spaces']),
        'Min epochs per cell': cfg['min_epochs'],
    }
    if cfg['do_fit']:
        # Tool 6 already rejects epochs on a 1/f criterion, so "why do some spectra still fail?"
        # is the first question this line raises. Answer it in place, and name the settings that
        # differ from the tool-6 run when its sidecar is available.
        sm = cfg['smooth']
        applied = ([f"median {sm['median_span_hz']:g} Hz"] if sm['enabled'] and sm['median'] else []) \
            + ([f"LOWESS {sm['lowess_span_hz']:g} Hz"] if sm['enabled'] and sm['lowess'] else [])
        smooth_desc = ((' then '.join(applied) if applied else 'none')
                       + ' - the band powers use the raw PSD')
        diffs = fit_settings_diff(cfg, side)
        fit_note = ('Some failures are expected even though tool 6 already flagged on a 1/f '
                    'criterion: tool 6 flags each (epoch, channel) pair while tools 7/7bis decide '
                    'per epoch (an epoch flagged on one channel out of four can legitimately be '
                    'kept), and the fit is recomputed here with this notebook&#39;s own PSD and '
                    'specparam settings.')
        if diffs:
            fit_note += ' Settings that differ from the tool-6 run: ' + '; '.join(diffs) + '.'
        elif side['found']:
            fit_note += ' The 1/f settings match the tool-6 run.'
        params.update({
            '1/f fit': (f"specparam, {cfg['fit']['fit_fmin']:g}-{cfg['fit']['fit_fmax']:g} Hz, "
                        f"mode {cfg['fit']['aperiodic_mode']}, peak width "
                        f"{cfg['fit']['peak_width_min']:g}-{cfg['fit']['peak_width_max']:g} Hz, "
                        f"max {cfg['fit']['max_n_peaks']} peaks, min height "
                        f"{cfg['fit']['min_peak_height']:g}, threshold {cfg['fit']['peak_threshold']:g} SD"),
            'Fit quality bounds': (f"R2 >= {cfg['fit']['r2_min']:g} and MAE <= {cfg['fit']['mae_max']:g}"
                                   f" - {np.nanmean(res['fit_ok']) * 100:.0f} % of spectra pass"
                                   + (', poor fits excluded from the aggregates'
                                      if cfg['exclude_bad_fit'] else ', all fits kept')
                                   + '<br><small style="color:#555">' + fit_note + '</small>'),
            'Smoothing (fit copy only)': smooth_desc,
        })
    params['Tool-6 provenance'] = (
        f"filter {side['filter']}, resample {side['resample']}, notch {side['notch']}, "
        f"PSD smoothing {side['psd_smoothing']}" if side['found']
        else '<span style="color:#ef6c00">no sidecar found - the preprocessing settings are unknown</span>')

    # The numbers themselves live in the TSV tables - the report only carries what helps read
    # them: the settings used and how many epochs stand behind each cell.
    n_nan = int(agg.get('n_epochs', pd.Series(dtype=int)).lt(cfg['min_epochs']).sum()) if len(agg) else 0
    items += [
        ('html', 'Parameters used', params_html(params)),
        ('html', 'Coverage - clean epochs per stage and channel',
         coverage_html(agg, ch_names)
         + f'<p><small>{n_nan} (stage x channel x band) cells fall below the '
           f'{cfg["min_epochs"]}-epoch minimum and are written as NaN. The values themselves are in '
           f'<code>{fid}_spectral_stage.tsv</code>.</small></p>'),
    ]
    save_report_html(out_reports / f'{fid}_spectral_report.html',
                     f'{fid} - spectral features', items)
    close_figs(items)
    phase('done', COST_REPORT)
    log(f'  wrote {len(agg)} aggregated rows to <code>{out_data}</code>')
    return True


def _pad_channel_union(df, ch_union, extra_keys):
    """Add explicit NaN rows for the channels a participant does not carry.

    Tools 7/7bis drop bad channels per participant, so the per-file tables have different channel
    sets. Padding to the union keeps the database table rectangular: a channel dropped for one
    participant is a visible `n_epochs = 0` / NaN row rather than a silently missing one."""
    if df is None or len(df) == 0 or not ch_union or 'channel' not in df.columns:
        return df
    blocks = []
    for fid, sub in df.groupby('file_id', sort=False):
        missing = [c for c in ch_union if c not in set(sub['channel'].astype(str))]
        if not missing:
            continue
        combos = sub.drop_duplicates(extra_keys)[extra_keys]
        for ch in missing:
            block = combos.copy()
            block['file_id'] = fid
            if 'source' in sub.columns:
                block['source'] = sub['source'].iloc[0]
            block['channel'] = ch
            block['n_epochs'] = 0
            if 'n_fit_ok' in df.columns:
                block['n_fit_ok'] = 0
            blocks.append(block)
    if not blocks:
        return df
    out = pd.concat([df] + blocks, ignore_index=True)
    return out.sort_values(['file_id'] + extra_keys + ['channel']).reset_index(drop=True)


def rebuild_globals(cfg, log):
    """Rebuild every database-level table by globbing the per-file tables from disk (never from an
    in-memory list, so an interrupted-then-skipped participant is never lost)."""
    data_dir, reports_dir = S['data_dir'], S['reports_dir']
    reports_dir.mkdir(parents=True, exist_ok=True)
    ch_union = S.get('ch_union', [])
    out = {}
    # `psd_stage` is deliberately NOT padded to the channel union: one row per frequency bin
    # would multiply the padding by ~180.
    for key, pattern, gname, pad_keys in [
            ('bandpower_stage', '*_spectral_stage.tsv', 'global_spectral_stage.tsv', ['stage', 'band']),
            ('aperiodic_stage', '*_aperiodic_stage.tsv', 'global_aperiodic_stage.tsv', ['stage']),
            ('psd_stage', '*_psd_stage.tsv', 'global_psd_stage.tsv', None)]:
        frames = []
        for f in sorted(data_dir.rglob(pattern)):
            if norm(f.name).startswith(norm('global_')):
                continue
            try:
                # file_id forced to str: a numeric participant id ("73") would otherwise be read
                # back as an int, giving a mixed int/str column that breaks sorting and matching.
                frames.append(pd.read_csv(f, sep='\t', dtype={'file_id': str}))
            except Exception as exc:
                log(f'&#9888; could not read {f.name}: {exc}')
        df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        if pad_keys:
            n_before = len(df)
            df = _pad_channel_union(df, ch_union, pad_keys)
            if len(df) > n_before:
                log(f'  {gname}: {len(df) - n_before} NaN rows added for the channels missing '
                    f'from some participants')
        df = _join_subject_info(df)
        out[key] = df
        _write_tsv(df, reports_dir / gname)
    return out


def write_excel(tables, cfg, failed, path, log):
    """One workbook, one sheet per table. Sheets over Excel's row limit are truncated with a
    pointer to the full TSV, which stays the source of truth."""
    if not HAS_OPENPYXL:
        log('&#9888; openpyxl is missing - the .xlsx workbook was not written (the TSV tables are complete)')
        return None
    cov = pd.DataFrame({'channel': S.get('ch_union', []),
                        'participants': [S.get('ch_counts', {}).get(c, 0) for c in S.get('ch_union', [])]})
    par = pd.DataFrame({'parameter': ['psd_method', 'psd_fmin_hz', 'psd_fmax_hz', 'welch_window_s',
                                      'welch_overlap_pct', 'welch_window', 'multitaper_bandwidth_hz',
                                      'aperiodic_fit', 'fit_fmin_hz', 'fit_fmax_hz', 'aperiodic_mode',
                                      'peak_width_limits_hz', 'min_peak_height', 'max_n_peaks',
                                      'peak_threshold_sd', 'fit_r2_min', 'fit_mae_max',
                                      'exclude_poor_fits', 'smoothing_fit_copy_only', 'bands',
                                      'measures', 'stages', 'epoch_averaging_space',
                                      'min_epochs_per_cell', 'night_thirds', 'source'],
                        'value': [cfg['psd']['method'], cfg['psd']['fmin'], cfg['psd']['fmax'],
                                  cfg['psd']['win_s'], cfg['psd']['overlap_pct'], cfg['psd']['window'],
                                  cfg['psd']['bandwidth'], cfg['do_fit'], cfg['fit']['fit_fmin'],
                                  cfg['fit']['fit_fmax'], cfg['fit']['aperiodic_mode'],
                                  f"{cfg['fit']['peak_width_min']}-{cfg['fit']['peak_width_max']}",
                                  cfg['fit']['min_peak_height'], cfg['fit']['max_n_peaks'],
                                  cfg['fit']['peak_threshold'], cfg['fit']['r2_min'],
                                  cfg['fit']['mae_max'], cfg['exclude_bad_fit'],
                                  str(cfg['smooth']), '; '.join(f'{n} {lo}-{hi}' for n, lo, hi in cfg['bands']),
                                  ', '.join(cfg['measures']), ', '.join(cfg['stages']),
                                  ', '.join(cfg['spaces']), cfg['min_epochs'], cfg['thirds'],
                                  S.get('source', '')]})
    sheets = {'bandpower_stage': tables.get('bandpower_stage'),
              'aperiodic_stage': tables.get('aperiodic_stage'),
              'psd_stage': tables.get('psd_stage'),
              'coverage': cov, 'parameters': par,
              'failed': pd.DataFrame(failed, columns=['file_id', 'reason']) if failed else pd.DataFrame()}
    try:
        with pd.ExcelWriter(path, engine='openpyxl') as xl:
            for name, df in sheets.items():
                if df is None or len(df) == 0:
                    pd.DataFrame({'note': ['(empty)']}).to_excel(xl, sheet_name=name, index=False)
                    continue
                if len(df) > EXCEL_MAX_ROWS:
                    log(f'&#9888; sheet "{name}" truncated to {EXCEL_MAX_ROWS} rows '
                        f'(of {len(df)}) - use the TSV for the full table')
                    df = df.head(EXCEL_MAX_ROWS)
                df.to_excel(xl, sheet_name=name, index=False)
        return path
    except Exception as exc:
        log(f'&#9888; could not write the workbook: {exc}')
        return None


def build_database_report(tables, cfg, path):
    """Database-level figures: group PSD, band power and aperiodic parameters across participants,
    closing with a "where do I look?" guide over the tables the run produced."""
    custom = S.get('custom_stages', [])
    stage_df = tables.get('bandpower_stage')
    psd_df = tables.get('psd_stage')
    items = [('fig', 'Group mean PSD per stage', plot_group_psd(psd_df, custom)),
             ('fig', 'Group PSD after removing the aperiodic component',
              plot_group_psd(psd_df, custom, value_col='psd_ap_removed_db',
                             ylabel='Power above the 1/f fit (dB)',
                             title='Group mean PSD per stage after removing the aperiodic component')
              if psd_df is not None and 'psd_ap_removed_db' in psd_df.columns else None),
             ('fig', 'Aperiodic parameters per stage',
              plot_group_aperiodic(tables.get('aperiodic_stage'), custom))]
    space = 'from_log' if 'log' in cfg['spaces'] else 'from_lin'
    for col, lab in [(f'power_db_{space}_mean', 'Power (dB/Hz)'),
                     ('power_rel_mean', 'Relative power'),
                     (f'power_ap_removed_db_{space}_mean', 'Power above the 1/f fit (dB)')]:
        fig = plot_group_band(stage_df, col, lab, custom)
        if fig is not None:
            items.append(('fig', f'{lab} across participants', fig))
    n_part = stage_df['file_id'].nunique() if stage_df is not None and len(stage_df) else 0
    # The sample-size figure sits with the summary, at the end: it is about how much data stands
    # behind the figures above, not a result of its own.
    items += [('html', 'Summary', params_html({
        'Participants in the tables': n_part,
        'Channels (union)': ', '.join(S.get('ch_union', [])),
        'Stages': ', '.join(cfg['stages']),
        'Source': S.get('source', ''),
    })),
        ('fig', 'Clean epochs per participant', plot_group_counts(stage_df)),
        ('html', 'Where to look', guidance_html(cfg, S['data_dir'], S['reports_dir'],
                                                has_subject_info=S.get('subj_info') is not None))]
    save_report_html(path, 'Spectral features - database report', items)
    close_figs(items)
    return path


def run(_=None):
    """Section-4 entry point: loop over the selected participants, then rebuild the database tables."""
    with out_run:
        clear_output()
        messages = []

        def log(msg):
            messages.append(msg)
            display(HTML(msg))

        try:
            if not S.get('parts'):
                lbl_run.value = '<span style="color:#c62828">Run the scan in Section 1 first.</span>'
                return
            cfg, errors = get_params()
            if errors:
                lbl_run.value = ('<span style="color:#c62828">' +
                                 '<br>'.join('&#10007; ' + e for e in errors) + '</span>')
                return
            todo = selected_participants()
            n_skipped = sum(1 for p in S['parts']
                            if part_checkboxes.get(p['file_id']) and
                            part_checkboxes[p['file_id']].value and p.get('done')) if cb_skip.value else 0
            if not todo:
                lbl_run.value = ('<span style="color:#ef6c00">Nothing to do &mdash; every selected '
                                 'participant is already processed (untick Skip to redo them).</span>')
            progress_part.max = max(len(todo), 1)
            progress_part.value = 0
            lbl_run.value = f'<i>Running {len(todo)} participants...</i>'

            done, failed = [], []
            for i, p in enumerate(todo, 1):
                progress_part.value = i - 1
                progress_part.description = f'Participants: {i}/{len(todo)}'
                log(f'<b>{p["file_id"]}</b>')
                try:
                    process_participant(p, cfg, log)
                    p['done'] = True
                    done.append(p['file_id'])
                except Exception as exc:
                    failed.append((p['file_id'], f'{type(exc).__name__}: {exc}'))
                    log(f'  <span style="color:#c62828">&#10007; failed: {exc}</span>')
                progress_part.value = i

            # The database-level outputs are non-fatal: every per-participant table is already on
            # disk, so a failure here (a locked workbook, a path over the Windows 260-character
            # limit) must not swallow the run summary.
            lbl_phase.value = '<small>rebuilding the database tables...</small>'
            reports_dir = S['reports_dir']
            tables, xlsx = {}, None
            try:
                tables = rebuild_globals(cfg, log)
                _write_tsv(pd.DataFrame(failed, columns=['file_id', 'reason']),
                           reports_dir / 'spectral_features_failed.tsv')
            except Exception as exc:
                log(f'&#9888; could not rebuild the database tables: {exc}')
            try:
                xlsx = write_excel(tables, cfg, failed,
                                   reports_dir / 'spectral_features_database.xlsx', log)
            except Exception as exc:
                log(f'&#9888; could not write the workbook: {exc}')
            try:
                build_database_report(tables, cfg, reports_dir / 'spectral_database_report.html')
            except Exception as exc:
                log(f'&#9888; could not write the database report: {exc}')
            lbl_phase.value = ''

            stage_df = tables.get('bandpower_stage')
            n_part = stage_df['file_id'].nunique() if stage_df is not None and len(stage_df) else 0
            log(f'<hr><b>{len(done)} processed</b>, {len(failed)} failed, {n_skipped} skipped &mdash; '
                f'the database tables now cover <b>{n_part}</b> participants.')
            log('<div style="background:#e8f5e9;border-left:4px solid #2e7d32;padding:8px 12px;'
                'margin:8px 0"><b>Start here:</b> open '
                f'<code>{reports_dir / "spectral_database_report.html"}</code>. Its closing '
                '<i>Where to look</i> section maps every table this run produced &mdash; which file '
                'answers which question, which column to read, and what to check before trusting a '
                'value.</div>')
            log(f'Data: <code>{S["data_dir"]}</code><br>Reports: <code>{reports_dir}</code>'
                + (f'<br>Workbook: <code>{xlsx}</code>' if xlsx else ''))
            if stage_df is not None and len(stage_df):
                summary = (stage_df[stage_df['stage'].isin(cfg['stages'])]
                           .drop_duplicates(['file_id', 'stage', 'channel'])
                           .groupby('file_id')['n_epochs'].sum().reset_index()
                           .rename(columns={'n_epochs': 'epoch x channel cells'}))
                display(HTML(summary.to_html(index=False, border=0)))
            lbl_run.value = (f'<span style="color:#2e7d32">Done &mdash; {len(done)} processed, '
                             f'{len(failed)} failed.</span>')
            build_participant_list()
        except Exception as exc:
            lbl_run.value = f'<span style="color:#c62828">Run failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


btn_run.on_click(run)
display(widgets.VBox([widgets.HBox([btn_run, lbl_run]),
                      progress_part, widgets.VBox([progress_step, _STEP_LEGEND]), lbl_phase, out_run]))

## Once the run is finished

**Start with the database report**, `reports_features_spectral/spectral_database_report.html`. Besides
the group figures, it closes with a **Where to look** section that maps everything this tool wrote:
which file answers which question, which column to read, what to check before trusting a value, how to
redraw any figure from the tables, and how to regroup participants by population variables.

Then open a participant's own `{file_id}_spectral_report.html` when you want to see what one recording
looks like, and read the numbers from `global_spectral_stage.tsv` (or the Excel workbook).